# YipNET: a coyote call classifier built on BirdNET

This notebook turns each clip into numbers with **BirdNET**, a model built for wildlife recorders: distant calls,
outdoor noise, long recordings. Its 6,522 species include **Coyote (Canis latrans)**. Small
models trained on BirdNET's numbers then answer the team's questions about each clip, ending with a
spreadsheet of each clip's best guess of its sounds, with an honest confidence, for a wildlife research group studying coyotes in the Northeast US to check.

**Source.** BirdNET is made by the K. Lisa Yang Center for Conservation Bioacoustics at the Cornell Lab
of Ornithology and Chemnitz University of Technology. It is installed with `pip install birdnet`
(version 1.4.0, from https://pypi.org/project/birdnet/, code at https://github.com/birdnet-team/birdnet),
which downloads the official BirdNET acoustic model v2.4 the first time it runs.

**License.** The BirdNET code is MIT-licensed, but the **model is licensed CC BY-NC-SA 4.0
(non-commercial)**. Educational and research use is allowed. A company using it for paid work would
need permission from the BirdNET team first.

Citation: Kahl, S., Wood, C. M., Eibl, M., & Klinck, H. (2021). BirdNET: A deep learning solution for
avian diversity monitoring. *Ecological Informatics*, 61, 101236.

## At a glance: the final tool

**The end goal:** a spreadsheet with every clip's **best guess of its sounds** (Howl, Bark, Yip-Yap, Chorus, Whine,
Growl and Group, alone or mixed) and a **confidence that is honest** about how often such a guess is right, plus
**when** each sound happens and the **length and pitch of the longest howl**. Reviewers start with the least confident
clips and correct the tags; `learn_from_corrections` retrains the model on their corrections. Run it with
`best_guess_sheet(folder, file='best_guesses.xlsx')` (Section 8).

| Column | What it means |
|---|---|
| Best guess | the most likely mix of sounds, for example Howl + Bark |
| Confidence (%) | the chance the whole mix is exactly right |
| Least sure part | the sound the model is least sure about, and how sure |
| Howl / Bark / Yip-Yap / Chorus / Whine / Growl / Group chance (%) | each sound's chance of being in the clip |
| Sound times (approx.) | for each sound in the best guess, the 3-second stretches where it most likely happens |
| Longest howl (s), Longest howl starts | the longest continuous howl the pitch tracker finds inside the clip's Howl stretches (30% sound times), when the best guess has a howl ("No howl detected" otherwise) |
| Longest howl pitch (Hz) | its fundamental frequency (median over the howl), never below 100 Hz |
| Warning | no coyote heard, whole clip unfamiliar, or an unidentified chunk |
| Corrected tag | blank, for the reviewers |

**How good it is** (held-out clips, Section 8): the best guess is exactly right for **61.9%** of clips (always guessing Howl: 57.8%), and the confidence is honest overall (stated 63.1%) and close band by band (stated 85%: right 87%; stated 40%: right 32%). Checking the least confident half of the clips finds 80% of the wrong guesses. **On new labeled clips:** Test #1 (181 clips, Section 9) 66.9% exactly right; Test #2 (180 clips, Section 11) 63.3% without the chunk tags and 61.7% with them (within chance; always Howl about 57%). The final tool uses the chunk tags (Section 10): same accuracy, more Yip-Yap found and a more honest confidence.

**What it gets wrong:** it mostly guesses Howl. Pure Howl clips are right 95.2% of the time, but mixes rarely are (Howl + Bark 7%, Chorus + Group 20%, Howl + Group 50%), and on those clips the confidence is too high, because the model cannot tell which howl hides a bark. Corrections to mixes are what will improve it most.

**Limits**

- **Distance.** BirdNET hears distant, faint coyotes poorly (36% of clips never reach a Coyote score of 0.1), and the
  model tells call types apart much less well on those clips.
- **Timestamps and howl measurements are approximate.** The labels say which sounds are in a clip but not when, so the
  sound times and howl lengths cannot be checked against the labels yet; reviewers should treat them as pointers.
- **Warnings matter.** Clips with a warning are right much less often than the rest; listen to them.

## Final model: what to run

1. Run every cell from **Setup** to the end, in order (Restart and Run All). Later sections reuse the data, the review
   checks and the shared tools of earlier ones, so do not skip sections. A full run takes about an hour on
   a laptop (10-15 minutes once `birdnet_features.npz` is cached).
2. **The final tool:** `best_guess_sheet('my_folder', file='best_guesses.xlsx')` (Section 8) gives every WAV file in
   `my_folder` its best guess of the sounds, a confidence, the sound times and the longest howl, least confident first.
   After the reviewers fill in "Corrected tag", run `learn_from_corrections('best_guesses.xlsx')` (list every corrected
   sheet so far), then make the next batch's sheet.

## How to read this notebook

**The story.** Each stage builds on the one before:

1. **Sections 1 to 3: the data and BirdNET.** 845 labeled clips, turned into numbers by BirdNET and summarized over
   each clip's "coyote moments".
2. **Sections 4 and 5: can the model name a clip's main call?** One call type per clip. It is right for 67% of clips,
   below the 76% from always saying "Howl", and about a quarter of its "wrong" answers name another call that really
   is in the clip. So one label per clip is the wrong question.
3. **Section 6: shared setup.** The seven sounds, the review checks and the tools Sections 7 and 8 use.
4. **Section 7: Group / Lone howl / Other.** The narrower question the model answers best.
5. **Section 8: the final tool.** Every clip's best guess of its sounds with an honest confidence, the sound times and
   the longest howl, and a correction loop.
6. **Section 9: the test data.** The final tool, unchanged, on 181 labeled clips it never saw in training
   (`labels_test1.xlsx`): a test best-guess sheet and a training-vs-test table.
7. **Section 10: chunk tags.** 256 hand-tagged 3-second chunks added as training rows: chunks exactly right 35.5%
   against 17.6% without them; whole clips unchanged. The final tool uses them.
8. **Section 11: Test #2.** The final tool on a second batch of 180 new labeled clips: exactly right 63.3% without
   chunk tags and 61.7% with them (always "Howl": 56.7%).

**How each section is laid out:** the section heading and an **At a glance** box (the question, the method, the main
result of the saved run, and what later sections use), then the code cells and their outputs, each with a short line
above it saying what it does. The settings the final tool depends on are listed under **How the model works** in
Setup, and explained again in the code where they are used.

### Roadmap

| Section | Question | Main result (saved run) |
|---|---|---|
| 1 | What recordings and labels do we have? | 845 clips; Howl is 76% of them |
| 2 | Does BirdNET hear these coyotes? | Partly: 32% of clips reach a Coyote score of 0.5; 36% never reach 0.1 |
| 3 | How do we turn a clip into one row of numbers? | The coyote moments: 2,053 numbers per clip |
| 4 | Can a model name the main call? | Accuracy about 0.67 (always "Howl": 0.763) |
| 5 | Are the "wrong" answers really wrong? | About a quarter name a call that is in the clip |
| 6 | Shared setup | Seven sounds, review checks, test splits and tools |
| 7 | Group / Lone howl / Other | 82.3% right on the 90% of clips that pass the review checks (always "Lone howl": 71.5%) |
| 8 | `best_guess_sheet` (the final tool) | Best guess right for 61.9% of clips (always "Howl": 57.8%); confidence honest overall (stated 63.1%) |
| 9 | How does the final tool do on new, labeled clips? | Test (181 clips): exactly right 66.9% (95% range 59.7% to 73.3%; always "Howl": 57.5%), against 61.9% held out in training (6 splits); confidence honest (stated 66.5%) |
| 10 | Do hand-tagged 3-second chunks help? | Chunks exactly right 35.5% with chunk tags against 17.6% without; whole clips unchanged |
| 11 | Does the test result hold on a second batch? | Test #2 (180 clips): exactly right 63.3% without chunk tags, 61.7% with (always "Howl": 56.7%) |

Every approach the team tried and did not keep is listed in the README.

### What a "head" is

BirdNET is a large network trained by the Cornell Lab on thousands of species. We use it **as is** (frozen): it turns
each 3-second chunk of audio into 1,024 numbers that describe the sound. The **head** is the small model we put on top
of those numbers and train on our own labels; it is the only part that learns what a howl, a bark or a group sounds
like. Section 4 uses a **two-layer head** (a small neural network); Sections 7 and 8 use the **simpler head**, a
logistic regression. Swapping the head never changes BirdNET.

### Words used throughout

- **Clip**: one recording, one row of the spreadsheet. **Chunk**: a 3-second piece of a clip, BirdNET's unit.
- **Embedding**: BirdNET's 1,024 numbers describing a chunk. The heads learn from these.
- **Head**: see above.
- **Round, held out**: cross-validation. The clips are split into 5 parts, keeping each recorder and night together,
  and each part is scored once by a model trained on the other four. Every result is on held-out clips.
- **Split**: one way of making the 5 rounds. Sections 7 and 8 use 6 splits (3 seeds, grouped by recorder + night or by
  recorder only).
- **Inner rounds**: the same split repeated inside a round's training clips, so calibration is chosen without
  touching the held-out clips. **Out of fold**: a probability from a model that did not train on that clip.
- **Calibration, confidence**: adjusting probabilities so that 0.8 means right about 8 times in 10; the confidence is
  the calibrated chance that an answer is right.
- **Review checks**: no coyote heard, whole clip unfamiliar, unidentified chunk (Section 6).
- **Precision**: of the clips given an answer, the share that are right. **Recall**: of the clips that really have it,
  the share found.
- **AUC**: the chance that a random clip with the target is ranked above a random clip without it (0.5 is a coin
  flip, 1.0 is perfect).
- **Fundamental frequency (pitch)**: the lowest tone of a howl, the one the ear hears as its pitch.

## Setup

### Your data: where to put it

The recordings and label spreadsheets belong to the research group and are **not** in this repository. To run the
notebook on your own data, put these next to this notebook (same names, or change the names in the next cell **and**
in the cells that read them):

| Put here | What it is | Used in |
|---|---|---|
| `labels_train.xlsx` | training labels, one row per clip; columns `Original File`, `Call Type`, `Secondary Call Type`, `Third Call Type`, `Group`, `ARU or Location`, `Date` | everywhere |
| `audio/` | the training WAV files named in `Original File` (named `RECORDER-YYYYMMDD_HHMMSS.wav`) | everywhere |
| `labels_test1.xlsx` and `Test_Audio_Files/` | test labels (same columns) and test WAV files | Section 9 |
| `chunk_tags_pilot_2026-11-20_2.xlsx` (optional) | hand-tagged 3-second chunks (make a blank one with Section 10's `MAKE_CHUNK_SHEET = True`) | Section 10 |
| `test2/` (optional) | `labels_test2.xlsx` and `Test_2_Audio_Files/` | Section 11 |

**Only want to tag recordings with the trained model?** You do not need this notebook or any data: double-click
`START_YipNET_Windows.bat` (or `START_YipNET_Mac.command`) and drag WAV files onto the page.

**Next cell:** checks that the data is in place and sets the working directory.

In [ ]:
import os

#PROJECT_DIR: the folder that holds this notebook and your data (see the table above). Change it if your data is elsewhere
PROJECT_DIR = os.path.abspath(os.getcwd())

#YOUR DATA: the files and folders this notebook reads (rename here and in the cells that read them if yours differ)
REQUIRED_DATA = {'labels_train.xlsx': 'training labels (Section 1)',
                 'audio': 'training WAV files (Section 1)',
                 'labels_test1.xlsx': 'test labels (Section 9)',
                 'Test_Audio_Files': 'test WAV files (Section 9)'}
OPTIONAL_DATA = {'chunk_tags_pilot_2026-11-20_2.xlsx': 'chunk tags (Section 10; without it the final tool has no chunk tags)',
                 'test2': 'Test #2 (Section 11; skipped without it)'}

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
missing = [f"  {name}  ({what})" for name, what in REQUIRED_DATA.items() if not os.path.exists(name)]
for name, what in OPTIONAL_DATA.items():
    print(('found:      ' if os.path.exists(name) else 'not found:  ') + f"{name}  ({what})")
if missing:
    raise FileNotFoundError('Your data is not in place yet. Put these next to this notebook (see the table above):\n'
                            + '\n'.join(missing))

### Loading packages

Use a Python 3.13 environment with requirements.txt installed. It has both BirdNET and TensorFlow.

TensorFlow uses a GPU automatically when one is visible; the cell below reports which device it found.
On Windows a GPU is only visible when the notebook runs in WSL2 (TensorFlow 2.11 and later dropped native Windows GPU support).

In [ ]:
#these libraries help us manipulate data
import time
import numpy as np
import pandas as pd

#these split the data and score everything
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
import soundfile as sf

#this runs the pre-trained BirdNET encoder
import birdnet

#these build and train the small classification head
import keras
from keras import layers

#use the GPU for TensorFlow (the classification heads) when one is visible; otherwise run on the CPU.
#BirdNET itself runs through LiteRT on the CPU, and its output is cached in birdnet_features.npz.
import tensorflow as tf
GPUS = tf.config.list_physical_devices('GPU')
if GPUS:
    try:
        for gpu in GPUS:
            #take GPU memory as needed instead of reserving all of it at once
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as error:
        #this can only be set before TensorFlow first uses the GPU: restart the kernel to apply it
        print('Could not set GPU memory growth:', error)
    print(f"TensorFlow {tf.__version__}: using {len(GPUS)} GPU(s): " + ', '.join(g.name for g in GPUS))
else:
    print(f"TensorFlow {tf.__version__}: no GPU found, running on the CPU.")
    if os.name == 'nt':
        print('    TensorFlow 2.11 and later cannot use a GPU on native Windows; run this notebook in WSL2 to use one.')

import warnings
warnings.filterwarnings('ignore')

#fixed seeds, so that the notebook gives roughly the same answer every time
SEED = 1234
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

### How the model works

The choices the final tool (Section 8) depends on. Each is explained again in the code where it is used.

**Data and labels**

- Seven sounds: the call types **Howl, Bark, Yip-Yap, Chorus, Whine and Growl** (Whine and Growl are kept as their own
  call types, although they are rare), plus **Group** (more than one coyote).
- A clip's sounds are every call type in its Call Type, Secondary and Third columns, plus **Group** when the Group
  column is marked. **Every Chorus counts as both Howl and Group**: a chorus is coyotes howling together (the
  labelers wrote "Chorus" without listing Howl), and it means more than one coyote.

**BirdNET and the clip summary**

- BirdNET acoustic model v2.4 runs on the **original audio**, in **3-second chunks with no overlap**, with its own
  default frequency range. We keep each chunk's embedding and its **Coyote score**.
- Each clip is summarized by its **coyote moments**: the average and maximum BirdNET embedding over the chunks whose
  Coyote score is **at least a quarter of the clip's highest** (at least the strongest chunk), plus five calling numbers
  (bouts, seconds of coyote chunks, spread over the clip, highest and average Coyote score): 2,053 numbers per clip.

**The simpler head and calibration**

- The **simpler head** is a logistic regression on the coyote moments, with its regularization strength (C) chosen
  from **0.001, 0.01 and 0.1** on a grouped fifth of the training clips, by a log loss in which every outcome counts
  equally.
- Its probabilities are **calibrated with Platt scaling** (a logistic regression on the log probabilities), fitted on
  out-of-fold probabilities of the training clips.

**Review checks** (a Warning on the sheet)

- **No coyote heard:** the clip's highest BirdNET Coyote score is below **0.005**.
- **Whole clip unfamiliar:** the clip is farther from every training clip than **99%** of a held-back fifth of the
  training clips are.
- **Unidentified chunk:** a 3-second chunk is farther from every training chunk than **99.5%** of the held-back clips'
  chunks are.

**Testing**

- Every result is on held-out clips over **6 splits**: seeds 1, 2 and 3, each with clips kept together by **recorder +
  night** and by **recorder only** (33 recorders; recorders are taken in a random order set by the seed and each is put
  in the fold with the fewest clips so far). Inner rounds use the same kind of groups.
- The **final models** train on **all 845 clips**: a fifth (kept together by recorder and night) chooses C, and the
  calibration comes from out-of-fold probabilities over all clips.

**The best-guess sheet (Section 8)**

- One simpler head per sound, each calibrated with Platt scaling.
- The **best guess** is the most likely valid mix (at least one call; Howl and Group whenever Chorus): a mix's chance is the
  product of each sound's chance of being present or absent, rescaled over the valid mixes.
- The **confidence** is that chance recalibrated on the training clips' out-of-fold mixes: the chance that the whole mix
  is exactly right. Every clip gets a best guess; the review checks become a warning.
- **Sound times:** each sound's head is run on each of the clip's coyote-moment chunks on its own; a sound's times are
  the chunks where its chance is **50% or more** (joined into stretches), or its strongest chunk when none is.
- **Longest howl** (only when the best guess has a howl): the pyin pitch tracker (100 to 2,000 Hz) runs on the audio
  after steady background noise is removed; a frame counts as howling when pyin is at least 50% sure it is voiced; the
  longest howl is the longest run of such frames, bridging dips of up to 0.25 s. Its pitch is the median fundamental
  frequency over the run; nothing below **100 Hz** is reported.
- The sheet is least confident first, with a "How to correct" sheet and the clips' BirdNET numbers; it never
  overwrites a file. `learn_from_corrections` reports how often the best guesses were right on the checked clips, then
  retrains on the 845 clips plus every checked clip.

## 1. The recordings and their labels

The team's spreadsheet of labeled clips and their audio. Whine and Growl are kept as their own call types.

**At a glance**

- **Question:** What recordings and labels do we have?
- **Method:** Read `labels_train.xlsx`, tidy the call type spellings, and check that every audio file is there.
- **Result (saved run):** 845 clips, all with audio. Main call type: Howl 645, Chorus 88, Bark 65, Yip-Yap 39, Whine 5, Growl 3. Howl is 76% of the clips, so always saying "Howl" is right 76% of the time.
- **Used later:** Every section.

In [ ]:
dat = pd.read_excel('labels_train.xlsx')

#the spreadsheet spells the same call several ways ('Howl', ' Howl', 'howl'), so labels are cleaned first
LABEL_COLUMNS = ['Call Type', 'Secondary Call Type', 'Third Call Type']
for column in LABEL_COLUMNS:
    dat[column] = dat[column].str.strip().str.title()
dat['ARU or Location'] = dat['ARU or Location'].str.strip()

#Whine and Growl are kept as their own call types (both are rare)

#the main call types, in a fixed order, so that a class index means the same thing everywhere
CALL_TYPES = sorted(dat['Call Type'].unique())
TYPE_INDEX = {t: i for i, t in enumerate(CALL_TYPES)}
dat['y'] = dat['Call Type'].map(TYPE_INDEX)
y = dat.y.values
dat['group'] = dat['Group'].notna()

#the audio itself
AUDIO_DIR = 'audio'
dat['audio_path'] = [os.path.join(AUDIO_DIR, f.strip()) for f in dat['Original File']]
has_audio = dat.audio_path.map(os.path.exists)
print(f"{has_audio.sum()} of {len(dat)} clips found in {AUDIO_DIR}/")
dat = dat[has_audio].reset_index(drop=True)
y = dat.y.values
print(dat['Call Type'].value_counts().to_string())

## 2. Turning audio into numbers with BirdNET

BirdNET cuts each recording into 3-second chunks. For every chunk it returns:

- an **embedding**: 1,024 numbers describing the sound, which our models learn from;
- a **score for each of its 6,522 species**, from 0 to 1. We keep only the **Coyote** score.

BirdNET only names species; it does not know call types. Telling a howl from a bark is the job of the models we train
on its numbers (the heads, Sections 4, 7 and 8). The Coyote score is used to find **where in a clip the coyote is**
(Section 3) and to flag clips where BirdNET hears no coyote at all (Section 6).

The first run downloads the BirdNET model (about 77 MB) and takes several minutes. The results are saved to
`birdnet_features.npz` and reused after that.

**At a glance**

- **Question:** Does BirdNET, a model built for wildlife recorders, hear these coyotes?
- **Method:** BirdNET v2.4 cuts each clip into 3-second chunks (original audio, no overlap). Each chunk gives an embedding of 1,024 numbers and a 0-to-1 Coyote score. Saved in `birdnet_features.npz`. The table groups each clip's highest Coyote score by the **spreadsheet's** Call Type (BirdNET does not see the labels).
- **Result (saved run):** 15,188 chunks. Median of each clip's highest Coyote score, by the spreadsheet's Call Type: Chorus 0.74, Bark 0.43, Yip-Yap 0.30, Howl 0.16, Growl 0.001, Whine 0.000. 32% of clips reach 0.5 in some chunk; 36% never reach 0.1. BirdNET hears many of these coyotes only faintly, especially howls (often distant), and barely registers whines and growls.
- **Used later:** The embeddings feed every model. The Coyote score picks each clip's coyote moments (Section 3) and flags "no coyote heard" (Section 6).

In [ ]:
#BirdNET acoustic model v2.4 on the original audio,
#3-second chunks with no overlap, BirdNET's default frequency range
model = birdnet.load('acoustic', '2.4', 'tf', library='litert')
SPECIES = list(model.species_list)
KEPT_SPECIES = {'coyote': 'Canis latrans_Coyote'}
KEPT_INDEX = {short: SPECIES.index(name) for short, name in KEPT_SPECIES.items()}
CHUNK_SECONDS = model.get_segment_size_s()
BATCH_FILES = 50          # clips per batch, to keep memory use low
print(f"BirdNET v{model.get_version()}: {len(SPECIES)} species, {CHUNK_SECONDS:.0f}-second chunks, "
      f"{model.get_embeddings_dim()}-number embeddings")


def birdnet_batch(paths):
    """BirdNET's embedding and Coyote score for every chunk of each clip in `paths`.
    BirdNET works on several clips at once and may return them in any order, so results are matched
    back to their clips by file name."""
    encoded = model.encode(paths)
    predicted = model.predict(paths, top_k=None, default_confidence_threshold=0.0)
    enc_row = {os.path.basename(str(p)): i for i, p in enumerate(encoded.inputs)}
    pred_row = {os.path.basename(str(p)): i for i, p in enumerate(predicted.inputs)}

    results = []
    for path in paths:
        name = os.path.basename(path)
        i, j = enc_row[name], pred_row[name]
        chunks = ~encoded.embeddings_masked[i, :, 0]                  # the chunks that hold real audio
        emb = encoded.embeddings[i][chunks].astype('float16')
        ids, probs = predicted.species_ids[j][chunks], predicted.species_probs[j][chunks]
        scores = {short: np.where(ids == index, probs, 0).max(1) for short, index in KEPT_INDEX.items()}
        results.append((emb, scores))
    return results


def compute_birdnet(paths, cache_file='birdnet_features.npz'):
    """BirdNET output for every clip, read from `cache_file` when already computed."""
    names = [os.path.basename(p) for p in paths]
    if os.path.exists(cache_file):
        stored = np.load(cache_file)
        if list(stored['name']) == names:
            cuts, emb = stored['offset'], stored['emb']
            scores = {short: stored[short] for short in KEPT_INDEX}
            return [(emb[a:b], {s: v[a:b] for s, v in scores.items()}) for a, b in zip(cuts[:-1], cuts[1:])]

    started = time.time()
    results = []
    for start in range(0, len(paths), BATCH_FILES):
        results += birdnet_batch(paths[start:start + BATCH_FILES])
        print(f"    {len(results)} of {len(paths)} clips ({time.time() - started:.0f}s)")
    np.savez_compressed(cache_file, name=np.array(names),
                        offset=np.cumsum([0] + [len(e) for e, _ in results]),
                        emb=np.concatenate([e for e, _ in results]),
                        **{short: np.concatenate([s[short] for _, s in results]) for short in KEPT_INDEX})
    return results


clips = compute_birdnet(list(dat.audio_path))
print(f"{len(clips)} clips, {sum(len(e) for e, _ in clips)} chunks in all")

Before any training: how strongly does BirdNET hear a coyote in clips of each call type? For each clip we take its
**highest Coyote score** in any chunk, then group the clips by the spreadsheet's Call Type.

In [ ]:
peaks = pd.DataFrame({short: [s[short].max() for _, s in clips] for short in KEPT_INDEX})
peaks['call type'] = dat['Call Type'].values

print("Median of each clip's highest Coyote score, by the spreadsheet's Call Type\n")
print(peaks.groupby('call type')['coyote'].agg(['size', 'median']).rename(columns={'size': 'clips'}).round(3).to_string())
print(f"\nClips where BirdNET's Coyote score reaches 0.5 in at least one chunk: {(peaks.coyote >= 0.5).mean():.0%}")
print(f"Clips where it never reaches 0.1: {(peaks.coyote < 0.1).mean():.0%}")

## 3. Summarizing each clip

A clip can be a minute long with only a few seconds of calling; the rest is wind, silence or other animals. So each
clip is summarized over its **coyote moments**: the 3-second chunks where BirdNET hears a coyote most (a Coyote score of
at least a quarter of the clip's highest, and at least the strongest chunk). The summary is the average and the maximum
embedding over those chunks, plus five calling numbers: how many separate bouts of coyote chunks, how many seconds of
them, how spread out they are over the clip, and the highest and average Coyote score. That makes one row of 2,053
numbers per clip.

Two other summaries were tested and dropped (averaging every chunk of the clip, and that average plus the clip's
species scores): the coyote moments predicted the main call best. Both are listed in the README.

**At a glance**

- **Question:** How do we turn a clip's chunks into one row of numbers?
- **Method:** The coyote moments: average and maximum embedding over the chunks where BirdNET hears a coyote most, plus 5 calling numbers.
- **Result (saved run):** 2,053 numbers for each of the 845 clips.
- **Used later:** The input for every model.

In [ ]:
#keep chunks scoring at least a quarter of the clip's highest Coyote score, and at least the 1 strongest chunk
COYOTE_SHARE_OF_PEAK = 0.25

def coyote_chunks(coyote):
    """True for each chunk kept as a coyote moment."""
    keep = coyote >= COYOTE_SHARE_OF_PEAK * coyote.max()
    keep[np.argmax(coyote)] = True
    return keep


def calling_numbers(keep, coyote):
    """Bouts (log), seconds of coyote chunks (log), spread over the clip (0 to 1), highest and average
    Coyote score of the kept chunks."""
    where = np.flatnonzero(keep)
    bouts = 1 + int((np.diff(where) > 1).sum())
    spread = (where[-1] - where[0] + 1) / len(keep)
    return [np.log1p(bouts), np.log1p(keep.sum() * CHUNK_SECONDS), spread, coyote.max(), coyote[keep].mean()]


def coyote_moments(clips):
    rows = []
    for e, s in clips:
        keep = coyote_chunks(s['coyote'])
        kept = e[keep].astype('float32')
        rows.append(np.concatenate([kept.mean(0), kept.max(0), calling_numbers(keep, s['coyote'])]))
    return np.array(rows)


V_birdnet_moments = coyote_moments(clips)
print('coyote moments:', V_birdnet_moments.shape)

## 4. A first model: one main call per clip

The first question: can a head on BirdNET's numbers name each clip's main call type (the spreadsheet's Call Type)?

The **head** here is a small two-layer neural network: it takes the 2,053 coyote-moment numbers, passes them
through two hidden layers (256 and 128 units, ReLU, dropout 0.3), and outputs one probability per call type. BirdNET
itself is not retrained; only the head learns from our labels. Rarer call types get more weight in training (balanced
class weights), so the head cannot simply say "Howl" every time.

The head is trained and scored in 5 rounds of cross-validation, with each recorder and night kept together. Each score
is the average over the 5 rounds, with the lowest and highest round in brackets.

**At a glance**

- **Question:** Can a head on BirdNET's numbers predict the main call type?
- **Method:** A two-layer neural network head with balanced class weights, on the coyote moments; 5-fold cross-validation with each recorder and night kept together. Accuracy and balanced accuracy (each call type counts equally), averaged over the 5 rounds. Whine and Growl have too few clips to score reliably, so balanced accuracy is also shown without them.
- **Result (from an earlier run with Whine and Growl pooled):** accuracy 0.667 (0.609 to 0.722) against 0.763 for always saying "Howl"; balanced accuracy without the rare calls 0.454.
- **Used later:** Section 5 checks its wrong answers.

In [ ]:
#the two-layer head
def build_head(n_outputs, n_inputs):
    """The two-layer head: 256 and 128 ReLU units with dropout 0.3, then one softmax output per call type."""
    keras.utils.set_random_seed(SEED)
    head = keras.Sequential([
        layers.Input(shape=(n_inputs,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(n_outputs, activation='softmax'),
    ])
    head.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy')
    return head


#5 rounds for Sections 4 and 5 (clips kept together by recorder and night)
dat['night'] = (dat['ARU or Location'] + ' ' + dat['Date'].dt.strftime('%Y-%m-%d')).fillna(dat['Original File'])
N_FOLDS = 5
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
ROUNDS = []
for train_rows, score_rows in outer.split(dat, y, groups=dat.night):
    inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    fit_idx, stop_idx = next(inner.split(train_rows, y[train_rows], groups=dat.night.values[train_rows]))
    ROUNDS.append((train_rows[fit_idx], train_rows[stop_idx], score_rows))

#Whine and Growl have too few clips (5 and 3 as the main call) to score each round reliably
common_call = ~np.isin(y, [TYPE_INDEX['Whine'], TYPE_INDEX['Growl']])

def cross_validate_head(features, model_name):
    """Train and score the head in each of the 5 rounds. Returns each round's scores and every clip's
    prediction from the round that scored it."""
    pred = np.zeros(len(dat), dtype=int)
    scores = []
    for k, (fit_rows, stop_rows, score_rows) in enumerate(ROUNDS):
        X_round = StandardScaler().fit(features[fit_rows]).transform(features)
        counts = np.bincount(y[fit_rows], minlength=len(CALL_TYPES))
        weights = {i: len(fit_rows) / (len(CALL_TYPES) * max(c, 1)) for i, c in enumerate(counts)}
        head = build_head(len(CALL_TYPES), features.shape[1])
        head.fit(X_round[fit_rows], y[fit_rows], validation_data=(X_round[stop_rows], y[stop_rows]),
                 epochs=100, batch_size=32, verbose=0, class_weight=weights,
                 callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                                          restore_best_weights=True)])
        pred[score_rows] = head.predict(X_round[score_rows], verbose=0).argmax(1)
        common = score_rows[common_call[score_rows]]
        scores.append({'model': model_name, 'round': k + 1,
                       'accuracy': accuracy_score(y[score_rows], pred[score_rows]),
                       'balanced accuracy': balanced_accuracy_score(y[score_rows], pred[score_rows]),
                       'balanced accuracy without Whine and Growl': balanced_accuracy_score(y[common], pred[common])})
    return scores, pred


SCORE_NAMES = ['accuracy', 'balanced accuracy', 'balanced accuracy without Whine and Growl']

def average_and_range(values):
    """'average (lowest to highest)' over the 5 rounds."""
    return f"{values.mean():.3f} ({values.min():.3f} to {values.max():.3f})"

**Next cell:** trains the head in the 5 rounds and prints accuracy and balanced accuracy.

In [ ]:
MODEL_VERSIONS = {'BirdNET, coyote moments': V_birdnet_moments}

version_scores, version_pred = [], {}
for name, features in MODEL_VERSIONS.items():
    scores, version_pred[name] = cross_validate_head(features, name)
    version_scores += scores
    print(f"  {name}: done")

display(pd.DataFrame(version_scores).groupby('model', sort=False)[SCORE_NAMES].agg(average_and_range))
print(f"Always guessing 'Howl' would give an accuracy of {(dat['Call Type'] == 'Howl').mean():.3f}")

**Next cell:** counts the clips of each call type predicted right, and shows the confusion table.

In [ ]:
print('Clips of each call type predicted right, out of all 845 (each clip scored once)')
right = pd.DataFrame({name: [(pred[y == i] == i).sum() for i in range(len(CALL_TYPES))]
                      for name, pred in version_pred.items()}, index=CALL_TYPES)
right.insert(0, 'clips', np.bincount(y, minlength=len(CALL_TYPES)))
display(right)

best = 'BirdNET, coyote moments'
print(f"\n{best}, all clips (rows: true call type; columns: predicted)")
display(pd.crosstab(pd.Categorical(dat['Call Type'], CALL_TYPES),
                    pd.Categorical([CALL_TYPES[k] for k in version_pred[best]], CALL_TYPES),
                    rownames=['true'], colnames=['predicted'], dropna=False))

## 5. Are the "wrong" answers really wrong?

Many clips hold more than one call, but Section 4 names only one. The model may often hear a call that
really is in the clip, just not the one marked as the main Call Type.

Here every "wrong" answer is checked against the clip's other labels. It counts as **arguably
reasonable** when the predicted call is listed as the clip's Secondary or Third Call Type, or when the
model says Chorus for a clip marked Group (several coyotes calling).

**At a glance**

- **Question:** When the model misses the main call, does it name another call that really is in the clip?
- **Method:** Each wrong answer is checked against the clip's Secondary and Third call columns; Chorus for a clip marked Group also counts as reasonable.
- **Result (from an earlier run with Whine and Growl pooled):** Of 281 wrong answers, 45 name a secondary or third call and 32 say Chorus for a Group clip: 27% are arguably reasonable. Counting them as right, accuracy goes from 0.667 to 0.759.
- **Used later:** Leads to predicting every sound in the clip (Section 8).

In [ ]:
main_call = dat['Call Type'].values
other_calls = [{c for c in (s, t) if isinstance(c, str)}
               for s, t in zip(dat['Secondary Call Type'], dat['Third Call Type'])]

reasonable_rows = {}
for name in ['BirdNET, coyote moments']:
    predicted_call = np.array([CALL_TYPES[k] for k in version_pred[name]])
    wrong = predicted_call != main_call
    in_clip = np.array([p in o for p, o in zip(predicted_call, other_calls)])
    chorus_for_group = (predicted_call == 'Chorus') & dat['group'].values
    reasonable = wrong & (in_clip | chorus_for_group)
    reasonable_rows[name] = {
        'clips marked wrong': wrong.sum(),
        'predicted call is a secondary/third call': (wrong & in_clip).sum(),
        'predicted Chorus for a Group clip': (wrong & chorus_for_group & ~in_clip).sum(),
        'share of wrong that is arguably reasonable': round(reasonable.sum() / wrong.sum(), 3),
        'accuracy, main call only': round((~wrong).mean(), 3),
        'accuracy, counting reasonable answers as right': round((~wrong | reasonable).mean(), 3)}

display(pd.DataFrame(reasonable_rows))

About a quarter of the "wrong" answers name a call that the labels say is in the clip. The rest are
real errors, many of them on faint clips. Only 137 clips list a secondary call; if the labels do not
always list every call heard, some of the remaining errors may also be calls that are really there.

This suggests predicting **which calls are in each clip** (as in the Concat column) instead of one
main call per clip.

## 6. Shared setup: the sounds, the review checks and the model tools

Everything Sections 7 and 8 rely on, in one place:

1. **The seven sounds** and each clip's true mix from the labels.
2. **The review checks**: no coyote heard, whole clip unfamiliar, unidentified chunk. On the sheet they become the
   Warning column.
3. **The simpler head and its calibration**, used for every model from here on.
4. **The test splits**: 6 ways of making the 5 rounds, and each clip's review notes in every split.
5. **The final training split** for the models trained on all 845 clips, and the review checks for new recordings.

**At a glance**

- **Question:** What do the final models need, and how are they tested?
- **Method:** Seven sounds (every Chorus counts as both Howl and Group); the review checks (Coyote score below 0.005; nearest-neighbor distance to the training clips and chunks above the 99% and 99.5% marks of held-back clips); a logistic regression head with C chosen on held-back clips and Platt calibration; 6 splits (seeds 1 to 3, recorder + night and recorder only).
- **Result (saved run):** Seven sounds over 845 clips (before Chorus was counted as Howl; rerun to refresh): Howl 689, Group 179, Bark 130, Chorus 97, Yip-Yap 58, Whine 6, Growl 4; 18 different mixes (Howl alone 488, Bark + Howl 101, Chorus + Group 88). Review notes per split: about 48 clips no coyote heard, 27 with an unidentified chunk and 12 whole clip unfamiliar (about 10% of clips).
- **Used later:** Sections 7 and 8.

**Next cell:** the seven sounds and each clip's true mix.

In [ ]:
#seven sounds: the call types Howl, Bark, Yip-Yap, Chorus, Whine and Growl, plus Group (more than one coyote). A clip's
#sounds are every call type in its Call Type, Secondary and Third columns, plus Group when the Group column is marked;
#every Chorus also counts as Group, since a chorus means more than one coyote. With CHORUS_INCLUDES_HOWL, every Chorus
#also counts as Howl: a chorus is coyotes howling together, and the labelers wrote "Chorus" without listing Howl. The
#pitch tracker backs this up: Chorus clips the model tagged Howl have tonal calls as long as labeled howls (median
#2.2 s against 2.1 s). False goes back to the labels as written
CHORUS_INCLUDES_HOWL = True
TAGS = ['Howl', 'Bark', 'Yip-Yap', 'Chorus', 'Whine', 'Growl', 'Group']
CALLS = [t for t in TAGS if t != 'Group']
dat['any_group'] = dat['group'] | (dat[LABEL_COLUMNS] == 'Chorus').any(axis=1)

#T marks, for every clip, which of the seven sounds the labels say are present (1) or not (0)
T = np.array([[(tag == 'Group' and group) or (tag != 'Group' and tag in set(calls)) for tag in TAGS]
              for calls, group in zip(dat[LABEL_COLUMNS].values, dat.any_group)], dtype='float32')
if CHORUS_INCLUDES_HOWL:
    print(f"Chorus clips without Howl in the labels, now counted as Howl: "
          f"{int(((T[:, TAGS.index('Chorus')] == 1) & (T[:, TAGS.index('Howl')] == 0)).sum())}")
    T[T[:, TAGS.index('Chorus')] == 1, TAGS.index('Howl')] = 1
true_sets = [frozenset(t for t, on in zip(TAGS, row) if on) for row in T]


def labeled(tag):
    """True for each clip whose labels list this sound."""
    return T[:, TAGS.index(tag)] == 1


print('Clips with each sound present\n')
print(pd.Series(T.sum(0).astype(int), index=TAGS).to_string())
print(f"\n{len(set(true_sets))} different combinations of sounds; the most common:")
print(pd.Series([' + '.join(sorted(s)) for s in true_sets]).value_counts().head(8).to_string())
print(f"\nChorus clips not marked Group in the spreadsheet, now counted as Group: "
      f"{((dat['Call Type'] == 'Chorus') & ~dat['group']).sum()}")

**Next cell:** the review checks.

In [ ]:
#every chunk of every clip, with the clip it belongs to and its position in that clip
chunk_emb = np.concatenate([e for e, _ in clips]).astype('float32')
chunk_clip = np.concatenate([np.full(len(e), i) for i, (e, _) in enumerate(clips)])
chunk_pos = np.concatenate([np.arange(len(e)) for e, _ in clips])

#no coyote heard: the clip's highest BirdNET Coyote score is below this
NO_COYOTE_BELOW = 0.005
#unidentified chunk: farther from every training chunk than 99.5% of the held-back clips' chunks are
CHUNK_NOVELTY_PERCENTILE = 99.5
#whole clip unfamiliar: farther from every training clip than 99% of the held-back clips are
CLIP_NOVELTY_PERCENTILE = 99


def unidentified_chunks(fit_rows, stop_rows):
    """For each chunk: True when it is unlike every chunk of the training clips (fit_rows), with the cutoff set on the
    held-back clips (stop_rows). Also returns the nearest-neighbor model and the cutoff, for new recordings."""
    in_fit = np.isin(chunk_clip, fit_rows)
    nearest = NearestNeighbors(n_neighbors=1, metric='cosine').fit(chunk_emb[in_fit])
    distance = np.zeros(len(chunk_emb))
    distance[~in_fit] = nearest.kneighbors(chunk_emb[~in_fit])[0][:, 0]
    cutoff = np.percentile(distance[np.isin(chunk_clip, stop_rows)], CHUNK_NOVELTY_PERCENTILE)
    return (distance > cutoff) & ~in_fit, nearest, cutoff


def seconds_label(position):
    """The time span of a chunk, for example '0:42-0:45'."""
    start = int(position * CHUNK_SECONDS)
    end = start + int(CHUNK_SECONDS)
    return f"{start // 60}:{start % 60:02d}-{end // 60}:{end % 60:02d}"


def review_notes(fit_rows, stop_rows):
    """The review checks for every clip ('' when none applies): no coyote heard, whole clip unfamiliar, unidentified
    chunks (with their times)."""
    X_novel = StandardScaler().fit(V_birdnet_moments[fit_rows]).transform(V_birdnet_moments)
    clip_distance = NearestNeighbors(n_neighbors=1, metric='cosine').fit(X_novel[fit_rows]).kneighbors(X_novel)[0][:, 0]
    unfamiliar = clip_distance > np.percentile(clip_distance[stop_rows], CLIP_NOVELTY_PERCENTILE)
    odd_chunk = unidentified_chunks(fit_rows, stop_rows)[0]
    notes = np.full(len(dat), '', dtype=object)
    for i in np.unique(chunk_clip[odd_chunk]):
        notes[i] = f"unidentified at {', '.join(seconds_label(p) for p in chunk_pos[(chunk_clip == i) & odd_chunk])}"
    notes[unfamiliar] = 'whole clip unfamiliar'
    notes[peaks.coyote.values < NO_COYOTE_BELOW] = 'no coyote heard'
    return notes

**Next cell:** the simpler head (logistic regression) for 2 or more outcomes, out-of-fold probabilities and calibration.

In [ ]:
#the simpler head: one logistic regression on the coyote moments. Its regularization strength (C) is chosen from these
#values on a held-back fifth of the training clips, by a log loss in which every outcome counts equally
LOGISTIC_C = [0.001, 0.01, 0.1]


def balanced_class_log_loss(truth, prob):
    """Log loss with every outcome counting equally."""
    prob = np.clip(prob, 1e-7, 1)
    return np.mean([-np.log(prob[truth == k, k]).mean() for k in range(prob.shape[1]) if (truth == k).any()])


def fit_logistic_outcomes(truth, fit_rows, stop_rows, features=V_birdnet_moments):
    """The simpler head for 2 or more outcomes: one logistic regression, its strength (C) chosen on the held-back
    (stopping) clips. Returns every clip's probability of each outcome."""
    X = StandardScaler().fit(features[fit_rows]).transform(features)
    best_loss, best = np.inf, None
    for C in LOGISTIC_C:
        fitted = LogisticRegression(C=C, max_iter=3000).fit(X[fit_rows], truth[fit_rows])
        loss = balanced_class_log_loss(truth[stop_rows], fitted.predict_proba(X[stop_rows]))
        if loss < best_loss:
            best_loss, best = loss, fitted
    return best.predict_proba(X)


def logistic_predictor(truth, fit_rows, stop_rows, features=V_birdnet_moments):
    """The simpler head for one yes/no target, as a function of new clip summaries (C chosen as above)."""
    scaler = StandardScaler().fit(features[fit_rows])
    X = scaler.transform(features)
    best_loss, best = np.inf, None
    for C in LOGISTIC_C:
        fitted = LogisticRegression(C=C, max_iter=3000).fit(X[fit_rows], truth[fit_rows])
        loss = balanced_class_log_loss(truth[stop_rows], fitted.predict_proba(X[stop_rows]))
        if loss < best_loss:
            best_loss, best = loss, fitted
    return lambda new: best.predict_proba(scaler.transform(new))[:, 1]


def out_of_fold(predict, train_rows, split, fit_mask=None):
    """Probabilities for the training clips, each from a model that did not train on it (inner rounds with
    the split's groups and seed). `predict(fit_rows, stop_rows)` returns probabilities for every clip;
    `fit_mask` limits the clips a model may train on (the held-out clips are always all predicted)."""
    prob = None
    for rest_idx, held_idx in group_folds(train_rows, split):
        rest, held = train_rows[rest_idx], train_rows[held_idx]
        if fit_mask is not None:
            rest = rest[fit_mask[rest]]
        fit_idx, stop_idx = group_folds(rest, split, n_splits=5)[0]
        p = predict(rest[fit_idx], rest[stop_idx])
        prob = np.zeros((len(dat),) + p.shape[1:]) if prob is None else prob
        prob[held] = p[held]
    return prob


def outcome_calibration(prob, truth):
    """Platt scaling for 2 or more outcomes: a logistic regression on the log probabilities, fitted on out-of-fold
    training probabilities, so that a calibrated 0.8 means right about 8 times in 10."""
    fitted = LogisticRegression(C=1.0, max_iter=3000).fit(np.log(np.clip(prob, 1e-6, 1)), truth)
    return lambda p: fitted.predict_proba(np.log(np.clip(p, 1e-6, 1)))

**Next cell:** the 6 test splits, each clip's review notes in every split, and the final training split (about two minutes).

In [ ]:
#6 splits: seeds 1, 2 and 3, each with two kinds of groups kept together: recorder + night (as in Section 4) and recorder
#only. With only 33 recorders of different sizes, sklearn's grouped splitter gives the same folds for every seed, so
#recorder-only folds take the recorders in a random order (set by the seed) and put each in the fold with the fewest
#clips so far
N_FOLDS = 5
SPLIT_SEEDS = [1, 2, 3]
FOLD_GROUPS = {'recorder + night': dat.night.values,
               'recorder only': dat['ARU or Location'].fillna(dat['Original File']).values}


def group_folds(rows, split, n_splits=N_FOLDS):
    """(rest, held) index pairs into `rows` for n_splits folds that keep each group together."""
    groups, seed = split['groups'][rows], split['seed']
    if split['kind'] == 'recorder + night':
        return list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
                    .split(rows, y[rows], groups=groups))
    fold_size, fold_of = np.zeros(n_splits), {}
    for name in np.random.default_rng(seed).permutation(pd.unique(groups)):
        fold_of[name] = fold_size.argmin()
        fold_size[fold_of[name]] += (groups == name).sum()
    fold = np.array([fold_of[name] for name in groups])
    return [(np.where(fold != f)[0], np.where(fold == f)[0]) for f in range(n_splits)]


def make_rounds(split):
    """5 rounds of (fit_rows, stop_rows, score_rows) with this split's seed and groups."""
    rounds = []
    for train_idx, score_idx in group_folds(np.arange(len(dat)), split):
        fit_idx, stop_idx = group_folds(train_idx, split, n_splits=5)[0]
        rounds.append((train_idx[fit_idx], train_idx[stop_idx], score_idx))
    return rounds


started = time.time()
SPLITS, REVIEW_NOTES = {}, {}
for fold, groups in FOLD_GROUPS.items():
    for seed in SPLIT_SEEDS:
        name = f"{fold}, seed {seed}"
        split = {'name': name, 'kind': fold, 'groups': groups, 'seed': seed}
        split['rounds'] = make_rounds(split)
        SPLITS[name] = split
        #each held-out clip's review note, from the round that holds it out
        notes = pd.Series('', index=np.arange(len(dat)), dtype=object)
        for fit_rows, stop_rows, score_rows in split['rounds']:
            notes.iloc[score_rows] = review_notes(fit_rows, stop_rows)[score_rows]
        REVIEW_NOTES[name] = notes
SPLIT_NAMES = list(SPLITS)

#the final models train on all 845 clips: a fifth of them (kept together by recorder and night) chooses C, and the
#review checks for new recordings are fitted on the same fifth
FINAL_SPLIT = {'kind': 'recorder + night', 'groups': dat.night.values, 'seed': SEED}
ALL_ROWS = np.arange(len(dat))
FINAL_FIT_ROWS, FINAL_STOP_ROWS = [ALL_ROWS[i] for i in group_folds(ALL_ROWS, FINAL_SPLIT, n_splits=5)[0]]


def train_review_checks(fit_rows, stop_rows):
    """The review checks for new recordings: what an unfamiliar clip and an unidentified chunk are, learned from
    `fit_rows` with the cutoffs set on `stop_rows`."""
    novelty_scaler = StandardScaler().fit(V_birdnet_moments[fit_rows])
    X_novel = novelty_scaler.transform(V_birdnet_moments)
    clip_nearest = NearestNeighbors(n_neighbors=1, metric='cosine').fit(X_novel[fit_rows])
    _, chunk_nearest, chunk_cutoff = unidentified_chunks(fit_rows, stop_rows)
    return {'novelty_scaler': novelty_scaler, 'clip_nearest': clip_nearest,
            'clip_cutoff': np.percentile(clip_nearest.kneighbors(X_novel[stop_rows])[0][:, 0], CLIP_NOVELTY_PERCENTILE),
            'chunk_nearest': chunk_nearest, 'chunk_cutoff': chunk_cutoff}


final_checks = train_review_checks(FINAL_FIT_ROWS, FINAL_STOP_ROWS)
print(f"6 splits and their review notes ready ({time.time() - started:.0f}s)")
print(f"Recorders: {pd.Series(FOLD_GROUPS['recorder only']).nunique()}; recorder + night groups: "
      f"{pd.Series(FOLD_GROUPS['recorder + night']).nunique()}")
print('\nClips held out in each round, for each split')
display(pd.DataFrame({name: [len(r[2]) for r in s['rounds']] for name, s in SPLITS.items()},
                     index=[f"round {k}" for k in range(N_FOLDS)]).T)
print('\nReview notes, mean clips per split')
display((pd.concat(REVIEW_NOTES.values()).str.replace(r' at .*', '', regex=True).replace('', '(none)')
         .value_counts() / len(SPLITS)).round(1).rename('clips per split').to_frame())

## 7. Group / Lone howl / Other

A narrower question than the full mix: is the clip **Group** (more than one coyote; a chorus counts), **Lone howl** (a
howl and no other coyote; other calls such as a bark are allowed) or **Other** (neither)?

- **One logistic regression with three outcomes** (the simpler head), calibrated on out-of-fold training probabilities.
- Every clip that passes the review checks gets the **most likely outcome** and its **confidence**; a clip that fails a
  check goes to **Review**.
- Tested on the **6 splits** (Section 6), including recorder-only splits, where the model never hears the held-out
  recorders in training.

**At a glance**

- **Question:** How well does the model tell Group, Lone howl and Other apart, and is its confidence honest?
- **Method:** A single 3-way logistic regression with calibration and the review checks, on the 6 splits.
- **Result (saved run):** 89.7% of clips pass the review checks and get an answer; **82.3%** of those answers are right (81.2% to 83.4% over the splits), against 71.5% for always saying Lone howl. Lone howl: precision 0.863, recall 0.858. Group: precision 0.706, recall 0.561. Other is rarely found (recall 0.08). Recorder-only splits are a little harder (0.819 vs 0.827). The confidence is honest overall (stated 0.832, right 0.823), a little high between 50% and 70%. 27 of every 100 answers leave another call in the clip unmentioned (mostly a bark in a Lone howl clip).
- **Used later:** Evidence only: it shows which part of the mix the model gets right most often. Section 8 does not use its results.

In [ ]:
#three outcomes: Group = more than one coyote (Chorus counts as Group); Lone howl = a howl and not Group (other calls
#such as a bark are allowed); Other = neither
OUTCOMES = ['Group', 'Lone howl', 'Other']
outcome_of = np.where(dat.any_group.values, 0, np.where(labeled('Howl'), 1, 2))   # index into OUTCOMES
truth_name = np.array(OUTCOMES)[outcome_of]
print(f"Outcomes over {len(dat)} clips:", ', '.join(f"{o} {(outcome_of == k).sum()}" for k, o in enumerate(OUTCOMES)))

**Next cell:** runs the 3-way model on the 6 splits (a few minutes).

In [ ]:
#the 3-way model is one logistic regression with three outcomes (the simpler head); calibration is a logistic regression
#on its log probabilities, fitted on out-of-fold training probabilities; the confidence is the top calibrated
#probability. The review checks send a clip to Review whatever the model says
def three_way_round(split, k):
    """The 3-way model's answer for the held-out clips of one round, as a DataFrame indexed by clip."""
    fit_rows, stop_rows, score_rows = split['rounds'][k]
    train_rows = np.concatenate([fit_rows, stop_rows])
    predict3 = lambda f, s: fit_logistic_outcomes(outcome_of, f, s)
    oof3 = out_of_fold(predict3, train_rows, split)[train_rows]
    cal3 = outcome_calibration(oof3, outcome_of[train_rows])
    p3 = cal3(predict3(fit_rows, stop_rows))
    notes = REVIEW_NOTES[split['name']]
    return pd.DataFrame({'round': k,
                         'answer': [('Review' if notes[i] else OUTCOMES[p3[i].argmax()]) for i in score_rows],
                         'confidence': [(np.nan if notes[i] else p3[i].max()) for i in score_rows],
                         'note': [notes[i] for i in score_rows]}, index=score_rows)


three_way_results = {}
started = time.time()
for name, split in SPLITS.items():
    three_way_results[name] = pd.concat([three_way_round(split, k) for k in range(N_FOLDS)]).sort_index()
    print(f"  {name}: done ({time.time() - started:.0f}s)", flush=True)

### How well does it do?

Measured on every clip, each scored once in the round where it was held out, for each of the 6 splits:

- **share answered**: clips that pass the review checks and get Group, Lone howl or Other.
- **accuracy**: of those, the share whose labels agree with the answer.
- **precision and recall** for each outcome.
- **base rate**: answering Lone howl for every clip, with no model.
- **other calls left unchecked per 100 answered**: answered clips whose labels list a call the answer does not
  mention (a bark in a clip answered Lone howl, for example).

In [ ]:
#"other calls left unchecked" counts answered clips whose labels list a call the answer does not mention: for Lone howl
#any call but Howl; for Group any call but Howl and Chorus; for Other every call
MENTIONED = {'Lone howl': {'Howl'}, 'Group': {'Howl', 'Chorus'}, 'Other': set()}
label_calls = [set(t) & set(CALLS) for t in true_sets]


def three_way_measures(r):
    answered = (r.answer != 'Review').values
    truth = truth_name[r.index]
    right = (r.answer.values == truth) & answered
    out = {'share answered': answered.mean(), 'accuracy (of answered)': right.sum() / answered.sum()}
    for o in OUTCOMES:
        mine = (r.answer == o).values
        out[f"{o}: precision"] = (truth[mine] == o).mean() if mine.any() else np.nan
        out[f"{o}: recall"] = (mine & (truth == o)).sum() / (truth == o).sum()
    out['base rate (answer Lone howl for every clip)'] = (truth == 'Lone howl').mean()
    unchecked = sum(bool(label_calls[i] - MENTIONED[a]) for i, a in zip(r.index[answered], r.answer.values[answered]))
    out['other calls left unchecked per 100 answered'] = 100 * unchecked / answered.sum()
    return out


per_split7 = pd.DataFrame({name: three_way_measures(three_way_results[name]) for name in SPLIT_NAMES})
summary7 = {}
for group, names in [('all 6 splits', SPLIT_NAMES), ('recorder + night (3 splits)', SPLIT_NAMES[:3]),
                     ('recorder only (3 splits)', SPLIT_NAMES[3:])]:
    part = per_split7[names]
    summary7[group] = [f"{m:.1f} ({lo:.1f} to {hi:.1f})" if 'per 100' in k else f"{m:.3f} ({lo:.3f} to {hi:.3f})"
                       for k, m, lo, hi in zip(part.index, part.mean(axis=1), part.min(axis=1), part.max(axis=1))]
print('Group / Lone howl / Other (held-out clips, each scored once per split): mean (lowest to highest split)')
display(pd.DataFrame(summary7, index=per_split7.index))

answers7 = np.concatenate([three_way_results[n].answer.values for n in SPLIT_NAMES])
truth7 = np.concatenate([truth_name[three_way_results[n].index] for n in SPLIT_NAMES])
print('\nWhat the labels say (rows) against the answer (columns), all 6 splits added up (each clip counted 6 times)')
display(pd.crosstab(pd.Series(truth7, name='labels'), pd.Series(answers7, name='answer'))
        .reindex(index=OUTCOMES, columns=OUTCOMES + ['Review'], fill_value=0))

#is the confidence honest? Within each band, how often the answers really are right
held7 = pd.concat([three_way_results[n] for n in SPLIT_NAMES])
held7 = held7[held7.answer != 'Review'].assign(right=lambda t: t.answer.values == truth_name[t.index])
held7['band'] = pd.cut(held7.confidence, [0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0001], right=False,
                       labels=['below 50%', '50 to 60%', '60 to 70%', '70 to 80%', '80 to 90%', '90% and above'])
print('\nIs the confidence honest? Answered held-out clips by stated confidence, all 6 splits added up')
display(held7.groupby('band', observed=True).agg(answers=('right', 'size'), stated_confidence=('confidence', 'mean'),
                                                  really_right=('right', 'mean')).round(3))
print(f"All answers: stated confidence {held7.confidence.mean():.3f} on average, really right {held7.right.mean():.3f}")

## 8. Best-guess spreadsheet with honest confidence (the final tool)

The end goal: for every clip, the model's **best guess of its sounds**, the mix of Howl, Bark, Yip-Yap, Chorus, Whine,
Growl and Group (like the spreadsheet's Concat column), with a **confidence that is honest** about how often such a
guess is right. For each sound in the guess, the sheet also says **when** it most likely happens, and for a howl, the
**length and pitch of the longest continuous howl**. Reviewers start with the least confident clips, correct the tags,
and the model learns from the corrections.

The cells, in order:

1. test the best guess and its confidence on the 6 splits;
2. check how honest the confidence is, by confidence band, by true mix and by sound, and what reviewing the least
   confident clips first would catch;
3. train the final models on all 845 clips;
4. the sound times and the longest howl, shown on a few training clips;
5. define `best_guess_sheet` and run it on the two made-up sounds;
6. define `learn_from_corrections`, which scores the best guesses against the reviewers' corrections and retrains.

**At a glance**

- **Question:** For each clip, what is the most likely mix of sounds, how honest is the model about its chance of being right, when does each sound happen, and how long and how low is the longest howl?
- **Method:** one calibrated simpler head per sound; the best guess is the most likely valid mix, its confidence recalibrated on training clips only; the same heads run on each 3-second chunk for the sound times; the pyin pitch tracker on noise-cleaned audio for the longest howl; a spreadsheet tool and a correction loop.
- **Result (saved run):** On held-out clips the best guess is exactly right for **61.9%** of clips (60.4% to 63.3% over the 6 splits), against 57.8% for always guessing Howl. The confidence is **honest overall** (stated 63.1% on average) and close band by band: stated 23% / 40% / 61% / 75% / 85% / 93%, really right 31% / 32% / 54% / 78% / 87% / 94%. Reviewing the least confident 20% of clips first finds 36% of the wrong guesses; the least confident half finds 80%. **The catch:** the model mostly guesses Howl: it is right for 95.2% of pure Howl clips, but rarely for mixes (Howl + Group 50%, Chorus + Group 20%, Howl + Bark 7%, Yip-Yap 2%, Whine 8%, Bark alone and Growl 0%), and within mixes the confidence is too high (Howl + Bark clips show 60%). Each sound on its own: Howl 82.9%, Bark 84.4%, Yip-Yap 93.2%, Chorus 88.4%, Whine 99.4%, Growl 99.4%, Group 86.2%. Clips with an unidentified chunk are right 28% of the time. Final models on all 845 clips: out of fold, right 63.8% at 63.8% stated. The sound times and the howl columns cannot be checked against the labels (no timed labels); the format and the pitch tracker were tested.
- **Note:** 61.9% is the model before chunk tags on the 6 splits; Section 10's 5-fold check gives 62.6% (no chunk tags) and 63.2% (with chunk tags).
- **Used later:** This is the final tool.

In [ ]:
import itertools

#the end goal is a spreadsheet with every clip's best guess of its
#sounds (the mix of Howl, Bark, Yip-Yap, Chorus, Whine, Growl and Group, as in the Concat column) and an honest confidence. One
#simpler head per sound (one logistic regression on the coyote moments, C chosen on the stopping clips), each calibrated
#with Platt scaling on out-of-fold training probabilities
#the best guess is the most likely valid mix: at least one call
#(Howl, Bark, Yip-Yap, Chorus, Whine or Growl), and Group whenever Chorus. A mix's chance is the product of each
#sound's chance of being present or absent, rescaled over the valid mixes. The confidence is that chance, recalibrated by a
#one-input logistic regression on its log-odds fitted on the training clips' out-of-fold mixes; it is the chance the whole
#mix is exactly right. Every clip gets a best guess; the review checks (no coyote heard, whole clip unfamiliar,
#unidentified chunk) become a warning, not a different tag. Tested on the 6 splits (Section 6)
CALLS20 = CALLS
MIXES20 = np.array([m for m in itertools.product([0, 1], repeat=len(TAGS))
                    if any(m[TAGS.index(c)] for c in CALLS20) and (not m[TAGS.index('Chorus')] or m[TAGS.index('Group')])
                    and (not CHORUS_INCLUDES_HOWL or not m[TAGS.index('Chorus')] or m[TAGS.index('Howl')])])
truth20 = (T == 1).astype(int)

#per-sound cutoffs: a sound goes into the best guess when its chance clears its own cutoff, instead of 50% for every
#sound. The sounds are not independent and the rare ones are seldom over 50%, so one shared cutoff is not always the most
#accurate. Each cutoff (20% to 80% in steps of 5%, for sounds with at least MIN_CLIPS_FOR_CUTOFF clips; 50% otherwise)
#is chosen on the training clips' out-of-fold chances to make the WHOLE best guess exactly right as often as possible,
#one sound at a time, three passes; a change is kept only if it gets more clips exactly right. The chances stay calibrated
#and the confidence is still recalibrated on the chosen mixes. (An earlier rule, the best F1 per sound, caught far more
#Bark and Yip-Yap but with many false alarms and fewer exactly right guesses; listed in the README.)
#USE_SOUND_CUTOFFS = False goes back to 50% for every sound
USE_SOUND_CUTOFFS = True
CUTOFF_GRID = np.round(np.arange(0.20, 0.801, 0.05), 2)
MIN_CLIPS_FOR_CUTOFF = 20


def choose_cutoffs(P, truth, force=False):
    """Each sound's cutoff (50% when switched off, unless `force`, or when the sound has too few clips)."""
    cutoffs = np.full(len(TAGS), 0.5)
    if not (USE_SOUND_CUTOFFS or force):
        return cutoffs
    exactly_right = lambda c: int((best_mix(P, c)[0] == truth).all(1).sum())
    best = exactly_right(cutoffs)
    for _ in range(3):
        changed = False
        for j in range(len(TAGS)):
            if truth[:, j].sum() < MIN_CLIPS_FOR_CUTOFF:
                continue
            for c in CUTOFF_GRID:
                trial = cutoffs.copy()
                trial[j] = c
                score = exactly_right(trial)
                if score > best:
                    best, cutoffs, changed = score, trial, True
        if not changed:
            break
    return cutoffs


def mix_name(m):
    return ' + '.join(t for t, on in zip(TAGS, m) if on)


def best_mix(P, cutoffs=None):
    """Best valid mix and its (unrecalibrated) chance for each row of per-sound chances P (clips x 6). With `cutoffs`,
    a sound counts as present once its chance clears its cutoff (on its own; the mix rules still apply); the returned
    chance is always the mix's real chance, so the confidence stays honest."""
    logp = np.log(np.clip(P, 1e-9, 1))[:, None, :] * MIXES20[None] + np.log(np.clip(1 - P, 1e-9, 1))[:, None, :] * (1 - MIXES20[None])
    joint = np.exp(logp.sum(2))
    joint /= joint.sum(1, keepdims=True)
    weight = np.zeros(len(TAGS)) if cutoffs is None else np.log((1 - np.asarray(cutoffs)) / np.asarray(cutoffs))
    k = (logp.sum(2) + (MIXES20 * weight).sum(1)[None]).argmax(1)
    return MIXES20[k], joint[np.arange(len(P)), k]


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def recalibration(raw, right):
    """The recalibration of the mix's chance, fitted on training clips."""
    fitted = LogisticRegression(C=1.0, max_iter=3000).fit(logit(raw)[:, None], right)
    return lambda r: fitted.predict_proba(logit(r)[:, None])[:, 1]


def sound_predict(truth, features=V_birdnet_moments):
    """The simpler head for one sound; when the fitting clips hold only one class (Whine and Growl are rare), a constant chance."""
    def predict(f, s):
        if truth[f].min() == truth[f].max():
            p = np.full(len(features), (truth[f].sum() + 0.5) / (len(f) + 1))
            return np.column_stack([1 - p, p])
        return fit_logistic_outcomes(truth, f, s, features=features)
    return predict


def safe_calibration(prob, truth):
    """Platt scaling, or no change when the training clips hold only one class."""
    return outcome_calibration(prob, truth) if truth.min() != truth.max() else (lambda p: p)


def sound_models(train_rows, split, fit_rows, stop_rows):
    """Per-sound held-out chance function and training out-of-fold chances (calibrated), for one round."""
    oof = np.zeros((len(dat), len(TAGS)))
    held = np.zeros((len(dat), len(TAGS)))
    for j in range(len(TAGS)):
        predict = sound_predict(truth20[:, j])
        raw_oof = out_of_fold(predict, train_rows, split)
        cal = safe_calibration(raw_oof[train_rows], truth20[train_rows, j])
        oof[train_rows, j] = cal(raw_oof[train_rows])[:, 1]
        held[:, j] = cal(predict(fit_rows, stop_rows))[:, 1]
    return oof, held


started = time.time()
guess20 = {}
compare20 = []                                                     # the same rounds with 50% for every sound
for name, split in SPLITS.items():
    parts = []
    for k in range(N_FOLDS):
        fit_rows, stop_rows, score_rows = split['rounds'][k]
        train_rows = np.concatenate([fit_rows, stop_rows])
        oof, held = sound_models(train_rows, split, fit_rows, stop_rows)
        #both versions, for the comparison: 50% for every sound, and per-sound cutoffs
        cut = choose_cutoffs(oof[train_rows], truth20[train_rows], force=True)
        mix1_tr, raw1_tr = best_mix(oof[train_rows], cut)
        recal1 = recalibration(raw1_tr, (mix1_tr == truth20[train_rows]).all(1))
        mix1_h, raw1_h = best_mix(held[score_rows], cut)
        mix0_tr, raw0_tr = best_mix(oof[train_rows])
        recal0 = recalibration(raw0_tr, (mix0_tr == truth20[train_rows]).all(1))
        mix0_h, raw0_h = best_mix(held[score_rows])
        compare20.append({'split': name, 'rows': score_rows, 'cutoffs': cut, 'after': mix1_h, 'before': mix0_h,
                          'conf after': recal1(raw1_h), 'conf before': recal0(raw0_h)})
        #the version in use
        mix_h, raw_h, recal = (mix1_h, raw1_h, recal1) if USE_SOUND_CUTOFFS else (mix0_h, raw0_h, recal0)
        parts.append(pd.DataFrame({'best guess': [mix_name(m) for m in mix_h], 'confidence': recal(raw_h),
                                   'raw chance': raw_h, 'right': (mix_h == truth20[score_rows]).all(1),
                                   'found all labeled': (mix_h >= truth20[score_rows]).all(1),
                                   **{f"{t} chance": held[score_rows, j] for j, t in enumerate(TAGS)}}, index=score_rows))
    r = pd.concat(parts).sort_index()
    notes = REVIEW_NOTES[name].reindex(r.index).fillna('')
    r['warning'] = np.where(notes.str.startswith('not sure'), '', notes.str.replace(r' at .*', '', regex=True))
    guess20[name] = r
    print(f"  {name}: done ({time.time() - started:.0f}s)")

all20 = pd.concat([guess20[n] for n in SPLIT_NAMES])
all20['truth'] = [mix_name(truth20[i]) for i in all20.index]
summary20 = pd.DataFrame({n: {'best guess exactly right': g.right.mean(), 'found every labeled sound': g['found all labeled'].mean(),
                              'stated confidence (mean)': g.confidence.mean(),
                              'always guess the commonest mix (Howl)': (np.array([mix_name(truth20[i]) for i in g.index]) == 'Howl').mean()}
                          for n, g in guess20.items()}).T
print('\nBest guess of the whole mix, held-out clips (each scored once per split)')
display(pd.concat([summary20, summary20.mean().rename('mean of the 6 splits').to_frame().T]).round(3))

**Next cell:** per-sound cutoffs, before and after, on the held-out clips of the 6 splits (the cutoffs are chosen on each round's training clips only). For each sound: labeled clips caught and false alarms (the sound guessed but not in the labels; with incomplete labels some of these may be real). For the whole mix: exactly right, found every labeled sound, and whether the stated confidence still matches how often the guess is right.

In [ ]:
#per-sound cutoffs, before (50% for every sound) and after, on the held-out clips (each clip scored once per split)
rows = []
for version in ('before', 'after'):
    caught = {t: 0 for t in TAGS}
    false = {t: 0 for t in TAGS}
    right, found, stated = [], [], []
    for r in compare20:
        truth = truth20[r['rows']]
        mix = r[version]
        for j, t in enumerate(TAGS):
            caught[t] += int(((mix[:, j] == 1) & (truth[:, j] == 1)).sum())
            false[t] += int(((mix[:, j] == 1) & (truth[:, j] == 0)).sum())
        right += list((mix == truth).all(1))
        found += list((mix >= truth).all(1))
        stated += list(r[f'conf {version}'])
    labeled = {t: int(sum(truth20[r['rows'], j].sum() for r in compare20)) for j, t in enumerate(TAGS)}
    rows.append({'version': version, **{f"{t} caught": f"{caught[t] / labeled[t]:.0%} ({false[t] // len(SPLIT_NAMES)} false)" for t in TAGS},
                 'exactly right': f"{np.mean(right):.1%}", 'found every labeled sound': f"{np.mean(found):.1%}",
                 'stated confidence': f"{np.mean(stated):.1%}"})
cutoffs_used = pd.DataFrame([r['cutoffs'] for r in compare20], columns=TAGS)
print('Cutoffs chosen in the held-out rounds (mean; 50% = unchanged):')
display((100 * cutoffs_used.mean()).round(0).to_frame('cutoff (%)').T)
print("Held-out clips, before and after (caught = share of the labeled clips with that sound; false = guessed but not "
      "labeled, per split):")
display(pd.DataFrame(rows).set_index('version').T)

**Next cell:** how honest the confidence is: by confidence band, by true mix, by sound and for the clips with a warning, and what reviewing the least confident clips first would catch.

In [ ]:
#how honest the confidence is, and what reviewing the least confident clips first would catch (held-out clips, all 6
#splits added up, so each clip counts up to 6 times)
BANDS20 = [0, 0.3, 0.5, 0.7, 0.8, 0.9, 1.0001]
BAND_NAMES20 = ['below 30%', '30 to 50%', '50 to 70%', '70 to 80%', '80 to 90%', '90% and above']
all20['band'] = pd.cut(all20.confidence, BANDS20, right=False, labels=BAND_NAMES20)
print('Is the confidence honest? Best guesses by stated confidence: how often the whole mix is really right')
display(all20.groupby('band', observed=True).agg(guesses=('right', 'size'), stated_confidence=('confidence', 'mean'),
                                                 really_right=('right', 'mean'))
        .assign(share_of_guesses=lambda t: t.guesses / t.guesses.sum()).round(3))

print('\nBy the true mix (the labels): how often the best guess is right, and its mean confidence')
by_mix = all20.groupby('truth').agg(clips=('right', 'size'), right=('right', 'mean'), confidence=('confidence', 'mean'))
by_mix['clips'] = by_mix['clips'] / len(SPLIT_NAMES)
display(by_mix.sort_values('clips', ascending=False).round(3))

print('\nEach sound on its own: share of clips where the best guess gets that sound right (present or absent)')
guessed = np.array([[t in g.split(' + ') for t in TAGS] for g in all20['best guess']])
display(pd.Series((guessed == truth20[all20.index].astype(bool)).mean(0), index=TAGS, name='right').round(3).to_frame())

print('\nClips with a warning (the review checks): how often the best guess is right')
display(all20.groupby(all20.warning.replace('', '(no warning)')).agg(
    clips_per_split=('right', lambda s: s.size / len(SPLIT_NAMES)), right=('right', 'mean'),
    confidence=('confidence', 'mean')).round(3))

order = np.argsort(all20.confidence.values, kind='stable')
wrong = ~all20.right.values[order]
print('\nReviewing the least confident clips first: share of the wrong best guesses found after checking the first ...')
display(pd.Series({f"{s:.0%} of clips": wrong[:int(round(s * len(order)))].sum() / wrong.sum()
                   for s in [0.1, 0.2, 0.3, 0.4, 0.5]}, name='wrong guesses found').round(3).to_frame())

**Next cell:** trains the final models on all 845 clips (a few seconds).

In [ ]:
#the final best-guess models train on all 845 clips the same way as
#the held-out test: one simpler head per sound (a fifth of the clips, from FINAL_SPLIT, chooses C), each calibrated on
#out-of-fold chances over all clips, and the recalibration of the mix's chance fitted on those out-of-fold mixes. The
#warnings use the review checks for new recordings (final_checks, Section 6)



def train_best_guess(features, truth, fit_rows, stop_rows, folds):
    """Per-sound predictors, their calibrations and the mix recalibration. `folds` is a list of (rest, held) row pairs
    for the out-of-fold chances; within each, a fifth of `rest` (by group) chooses C."""
    rows = np.arange(len(features))
    oof = np.zeros((len(features), len(TAGS)))
    predictors, calibrations = [], []
    for j in range(len(TAGS)):
        raw = np.zeros((len(features), 2))
        for rest, held, stop in folds:
            raw[held] = sound_predict(truth[:, j], features)(np.setdiff1d(rest, stop), stop)[held]
        cal = safe_calibration(raw, truth[:, j])
        oof[:, j] = cal(raw)[:, 1]
        if truth[fit_rows, j].min() == truth[fit_rows, j].max():
            const = (truth[fit_rows, j].sum() + 0.5) / (len(fit_rows) + 1)
            predictors.append(lambda new, c=const: np.full(len(new), c))
        else:
            predictors.append(logistic_predictor(truth[:, j], fit_rows, stop_rows, features=features))
        calibrations.append(cal)
    cutoffs = choose_cutoffs(oof, truth)
    mix, raw_mix = best_mix(oof, cutoffs)
    recal = recalibration(raw_mix, (mix == truth).all(1))
    def chance(new):
        out = []
        for p, cal in zip(predictors, calibrations):
            q = p(new)
            out.append(cal(np.column_stack([1 - q, q]))[:, 1])
        return np.column_stack(out)
    return {'chance': chance, 'recal': recal, 'oof right': (mix == truth).all(1), 'oof confidence': recal(raw_mix),
            'cutoffs': cutoffs}


def final_split_folds(rows):
    """FINAL_SPLIT's grouped folds over `rows`, each with a grouped fifth of the rest for choosing C."""
    out = []
    for rest_idx, held_idx in group_folds(rows, FINAL_SPLIT):
        rest = rows[rest_idx]
        _, stop_idx = group_folds(rest, FINAL_SPLIT, n_splits=5)[0]
        out.append((rest, rows[held_idx], rest[stop_idx]))
    return out


started = time.time()
final20 = train_best_guess(V_birdnet_moments, truth20, FINAL_FIT_ROWS, FINAL_STOP_ROWS, final_split_folds(ALL_ROWS))
print(f"Final best-guess models trained on all {len(dat)} clips ({time.time() - started:.0f}s); out of fold, the best guess "
      f"is right for {final20['oof right'].mean():.3f} of clips at a mean stated confidence of {final20['oof confidence'].mean():.3f}")
print('Cutoff per sound (%):', {t: int(round(100 * c)) for t, c in zip(TAGS, final20['cutoffs'])})

### When each sound happens, and the longest howl

The labels say which sounds are in a clip but not when, so the model was never taught times. To estimate them, each
sound's head is run on every 3-second chunk on its own (the chunk stands in for a whole clip's coyote moments). Only the clip's coyote moments (the chunks the
model learns from, Section 3) are considered. A sound's **times** are the coyote-moment chunks where its chance is 50% or
more, joined into stretches; when none reaches 50%, the strongest one is given, marked "(strongest)". A second, looser column does the same at
**30%**, so it also shows the quieter or less certain stretches of each sound. These times are **approximate** and cannot be checked against the labels.

For a clip whose best guess has a **howl**, the pyin pitch tracker follows the strongest tone between 100 and 2,000 Hz,
frame by frame, after steady background noise (hum, wind) is removed. A **run** is a stretch of frames pyin is at least
50% sure are voiced (a clear tone), bridging dips of up to 0.25 s. Pyin hears any tone (a bird, an insect, a siren), so
only runs that overlap the clip's **Howl stretches** in the looser (30%) sound times count; the **longest continuous
howl** is the longest of those, measured at its full length (not cut at the 3-second chunk edges). Its **pitch** is
the median fundamental frequency over that run; frequencies below 100 Hz are never reported. When the best guess has
no howl, the sheet says "No howl detected"; when tones were found but none in the Howl stretches, "Howl guessed, no
continuous howl in the howl times". (Before Nov 17 the longest run anywhere in the clip was measured; on the training
sheet, 35% of those runs lay outside every Howl stretch.)

**Next cell:** defines both, and shows the sound times on a few training clips (these clips were in training, so this
only illustrates the format).

In [ ]:
#sound times: each sound's head runs on every chunk on its own. A chunk's coyote-moments row is its embedding as both
#the average and the maximum, with the calling numbers of a single chunk (1 bout, 3 seconds, its share of the clip, and
#its Coyote score as the highest and average). Only the clip's coyote moments are considered (the chunks the model
#learns from; Section 3). A sound's times are the coyote-moment chunks where its chance is at least TIME_CHANCE, joined
#into stretches, or its strongest coyote-moment chunk when none is
TIME_CHANCE = 0.5
#the looser second column: the same rule at a 30% chance
LOOSE_TIME_CHANCE = 0.3


def chunk_summaries(emb, coyote):
    """One coyote-moments row per chunk, so a clip-level head can score each chunk."""
    e = emb.astype('float32')
    n = len(e)
    calling = np.column_stack([np.full(n, np.log1p(1)), np.full(n, np.log1p(CHUNK_SECONDS)), np.full(n, 1 / n),
                               coyote, coyote])
    return np.hstack([e, e, calling])


def stretch_label(first, last):
    """The time span from the start of chunk `first` to the end of chunk `last`, for example '0:12-0:21'."""
    start, end = int(first * CHUNK_SECONDS), int((last + 1) * CHUNK_SECONDS)
    return f"{start // 60}:{start % 60:02d}-{end // 60}:{end % 60:02d}"


def sound_times(emb, coyote, mix, model, chance=TIME_CHANCE):
    """'Howl 0:00-0:12, 0:24-0:27; Bark 0:42-0:45 (strongest)' for the sounds in `mix` (a 0/1 row over TAGS)."""
    P = model['chance'](chunk_summaries(emb, coyote))
    moment = coyote_chunks(coyote)
    parts = []
    for j, tag in enumerate(TAGS):
        if not mix[j]:
            continue
        on = np.flatnonzero((P[:, j] >= chance) & moment)
        if len(on):
            breaks = np.flatnonzero(np.diff(on) > 1)
            starts, ends = np.r_[on[0], on[breaks + 1]], np.r_[on[breaks], on[-1]]
            parts.append(f"{tag} " + ', '.join(stretch_label(a, b) for a, b in zip(starts, ends)))
        else:
            k = np.flatnonzero(moment)[P[moment, j].argmax()]
            parts.append(f"{tag} {stretch_label(k, k)} (strongest)")
    return '; '.join(parts)


#the longest continuous howl: pyin from 100 to 2,000 Hz on audio resampled to 8,000 Hz (128 ms frames every 32 ms), after
#steady background noise is removed (each frequency's median level over the clip, times 2, is subtracted). A frame is
#howling when pyin is at least 50% sure it is voiced; runs bridge dips of up to 0.25 s. Only runs that overlap the
#clip's Howl stretches (the 30% sound times, howl_spans) count, since pyin hears any tone (birds, insects, sirens); the
#longest howl is the longest of those, at its full length. Its pitch is the median fundamental frequency over its voiced
#frames, never below MIN_PITCH_HZ
PITCH_RATE_HZ, PITCH_FRAME, PITCH_HOP = 8000, 1024, 256
MIN_PITCH_HZ, MAX_PITCH_HZ = 100, 2000
VOICED_PROB, HOWL_GAP_SECONDS = 0.5, 0.25
HOP_SECONDS = PITCH_HOP / PITCH_RATE_HZ


def remove_steady_noise(audio):
    """Subtract twice each frequency's median level over the clip (hum, wind, distant insects)."""
    import librosa
    spec = librosa.stft(audio, n_fft=PITCH_FRAME, hop_length=PITCH_HOP)
    level = np.abs(spec)
    cleaned = np.maximum(level - 2 * np.median(level, axis=1, keepdims=True), 0) * np.exp(1j * np.angle(spec))
    return librosa.istft(cleaned, hop_length=PITCH_HOP, length=len(audio))


def howl_spans(emb, coyote, model, chance=LOOSE_TIME_CHANCE):
    """The clip's Howl stretches in seconds, [(start, end), ...], exactly as the 30% Sound times column shows them: the
    coyote-moment chunks where the Howl chance is at least `chance`, joined, or the strongest one when none is."""
    j = TAGS.index('Howl')
    P = model['chance'](chunk_summaries(emb, coyote))[:, j]
    moment = coyote_chunks(coyote)
    on = np.flatnonzero((P >= chance) & moment)
    if not len(on):
        on = np.flatnonzero(moment)[[P[moment].argmax()]]
    breaks = np.flatnonzero(np.diff(on) > 1)
    starts, ends = np.r_[on[0], on[breaks + 1]], np.r_[on[breaks], on[-1]]
    return [(float(a * CHUNK_SECONDS), float((b + 1) * CHUNK_SECONDS)) for a, b in zip(starts, ends)]


def voiced_runs(voiced, bridge_seconds):
    """(first, last) frame of every voiced run, bridging gaps up to bridge_seconds."""
    frames = np.where(voiced)[0]
    if not len(frames):
        return []
    breaks = np.where(np.diff(frames) > bridge_seconds / HOP_SECONDS + 1)[0]
    starts, ends = np.r_[frames[0], frames[breaks + 1]], np.r_[frames[breaks], frames[-1]]
    return list(zip(starts, ends))


def longest_run(voiced, bridge_seconds, spans=None):
    """(first, last) frame of the longest voiced run that overlaps one of `spans` (seconds; None = anywhere in the clip),
    or None when there is no such run."""
    runs = [(a, b) for a, b in voiced_runs(voiced, bridge_seconds)
            if spans is None or any(a * HOP_SECONDS < end and (b + 1) * HOP_SECONDS > start for start, end in spans)]
    return max(runs, key=lambda r: r[1] - r[0]) if runs else None


def longest_howl(path, spans=None):
    """The longest continuous howl in a WAV file that overlaps the Howl stretches `spans` (None = anywhere):
    {'seconds', 'start', 'pitch'} (pitch in Hz); {} when there are tones but none in `spans`; None when there is no tone."""
    import librosa
    audio, _ = librosa.load(path, sr=PITCH_RATE_HZ, mono=True)
    f0, _, prob = librosa.pyin(remove_steady_noise(audio), fmin=MIN_PITCH_HZ, fmax=MAX_PITCH_HZ, sr=PITCH_RATE_HZ,
                               frame_length=PITCH_FRAME, hop_length=PITCH_HOP)
    voiced = (prob >= VOICED_PROB) & np.isfinite(f0) & (f0 >= MIN_PITCH_HZ)
    if longest_run(voiced, HOWL_GAP_SECONDS) is None:
        return None
    run = longest_run(voiced, HOWL_GAP_SECONDS, spans)
    if run is None:
        return {}
    pitch = f0[run[0]:run[1] + 1][voiced[run[0]:run[1] + 1]]
    return {'seconds': (run[1] - run[0] + 1) * HOP_SECONDS, 'start': run[0] * HOP_SECONDS, 'pitch': float(np.median(pitch))}


def howl_columns(has_howl, path, spans=None):
    """The sheet's three howl columns for one clip; `spans` are its Howl stretches (howl_spans)."""
    if not has_howl:
        return 'No howl detected', '', ''
    m = longest_howl(path, spans)
    if m is None:
        return 'Howl guessed, no continuous howl measured', '', ''
    if not m:
        return 'Howl guessed, no continuous howl in the howl times', '', ''
    start = int(m['start'])
    return round(m['seconds'], 1), f"{start // 60}:{start % 60:02d}", round(m['pitch'])


#illustration on training clips (the final models trained on them): 4 clips whose best guess has two or more sounds, and
#2 whose best guess is a howl alone
all_mix, _ = best_mix(final20['chance'](V_birdnet_moments), final20.get('cutoffs'))
demo_rows = list(np.flatnonzero(all_mix.sum(1) >= 2)[:4]) + list(np.flatnonzero((all_mix.sum(1) == 1) & (all_mix[:, 0] == 1))[:2])
demo_mix = all_mix[demo_rows]
display(pd.DataFrame({'File': dat['Original File'].values[demo_rows],
                      'Labels': [' + '.join(sorted(true_sets[i])) for i in demo_rows],
                      'Best guess': [mix_name(m) for m in demo_mix],
                      'Sound times (approx.)': [sound_times(clips[i][0], clips[i][1]['coyote'], m, final20)
                                                for i, m in zip(demo_rows, demo_mix)]}))

**Next cell:** defines `best_guess_sheet` and runs it on the two made-up sounds.

**Next cell:** defines how the best-guess spreadsheet looks: a tidy "Best guesses" sheet (frozen header, filters, banding, number formats, the Corrected tag column highlighted), a **Summary** sheet of live Excel formulas (so it updates as reviewers fill in Corrected tag) and a **Column guide**. `best_guess_sheet` uses it every time it saves.

In [ ]:
#the best-guess workbook: pandas writes the data, then openpyxl tidies it and adds a Summary (live Excel formulas) and a
#Column guide. Sheets: Summary, Best guesses, How to correct, Column guide. Columns are looked up by their header, never
#by letter, and the data values are written untouched (formats are for display only)
import datetime
import math
import re

import pandas as pd
from openpyxl.comments import Comment
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

SHEET_GREEN, SHEET_BAND, SHEET_GRID, SHEET_INPUT, SHEET_NOTE = '2E4A36', 'F4F6F2', 'D9D9D9', 'FFF7D6', '595959'
SHEET_BANDS = [(None, 30, 'below 30%'), (30, 50, '30-50%'), (50, 70, '50-70%'), (70, 80, '70-80%'),
               (80, 90, '80-90%'), (90, None, '90%+')]
DATA_SHEET = 'Best guesses'
F_COUNT, F_PCT, F_CONF, F_SEC, F_HOURS, F_HZ = '#,##0', '0.0%', '0.0"%"', '0.0" s"', '0.00" h"', '0" Hz"'

_thin = Side(style='thin', color=SHEET_GRID)
_BOX = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
_FONT = Font(name='Calibri', size=11)
_HEAD_FONT = Font(name='Calibri', size=11, bold=True, color='FFFFFF')


def _fill(color):
    return PatternFill(start_color=color, end_color=color, fill_type='solid')


def _number_format(header):
    if header in ('Clip length (s)', 'Longest howl (s)'):
        return F_SEC
    if header in ('Confidence (%)', 'Match with labels (%)') or header.endswith('chance (%)'):
        return '0"%"'
    if header == 'Longest howl pitch (Hz)':
        return F_HZ
    return None


def _wrapped(header):
    return header.startswith('Sound times') or header in ('Least sure part', 'Warning', 'Labels in the spreadsheet')


def _is_input(header):
    return header.startswith('Corrected tag')


def _shown(value):
    """Roughly the characters a cell shows."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    if isinstance(value, float):
        return f"{value:.1f}  "
    return str(value)


def _is_number(value):
    return isinstance(value, (int, float)) and not (isinstance(value, float) and math.isnan(value))


def _print_setup(ws, title_rows=None, one_page=False):
    ws.page_setup.orientation = 'landscape'
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 1 if one_page else 0
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.print_options.gridLines = False
    if title_rows:
        ws.print_title_rows = title_rows


def _header_cell(cell, align='center', wrap=True):
    cell.font = _HEAD_FONT
    cell.fill = _fill(SHEET_GREEN)
    cell.border = _BOX
    cell.alignment = Alignment(wrap_text=wrap, vertical='center', horizontal=align)


def _style_best_guesses(ws, out, how_to_correct):
    headers = list(out.columns)
    n, last = len(out), max(len(out) + 1, 2)
    letter = {h: get_column_letter(i + 1) for i, h in enumerate(headers)}

    #widths sized to content with caps; long text columns wrap at about 45
    width = {}
    for h in headers:
        if _wrapped(h):
            width[h] = 45
        elif _is_input(h):
            width[h] = 36
        else:
            longest = max([len(_shown(v)) for v in out[h]] + [max(len(w) for w in str(h).split())])
            width[h] = min(max(longest + 2, 9), 30)
        ws.column_dimensions[letter[h]].width = width[h]

    for h in headers:
        _header_cell(ws[f"{letter[h]}1"])
    header_lines = max(math.ceil(len(h) / (width[h] * 1.05)) for h in headers)
    ws.row_dimensions[1].height = max(40, min(header_lines, 7) * 15 + 4)

    #data: text left, numbers right, the short yes/no column centered; formats for display only
    formats = {h: _number_format(h) for h in headers}
    for row in ws.iter_rows(min_row=2, max_row=n + 1, max_col=len(headers)):
        for h, cell in zip(headers, row):
            cell.font, cell.border = _FONT, _BOX
            number = _is_number(cell.value)
            horizontal = 'center' if h == 'Best guess right?' else 'right' if number else 'left'
            cell.alignment = Alignment(vertical='top', horizontal=horizontal, wrap_text=_wrapped(h) or _is_input(h))
            if number and formats[h]:
                cell.number_format = formats[h]
    for i in range(n):
        lines = 1
        for h in headers:
            if _wrapped(h):
                text = _shown(out[h].iloc[i])
                lines = max(lines, math.ceil(len(text) / (width[h] * 1.1)) if text else 1)
        if lines > 1:
            ws.row_dimensions[i + 2].height = 15 * min(lines, 4) + 2

    #Corrected tag: where to type (light yellow) and how (an input message; the full instructions on its header)
    corrected = next((h for h in headers if _is_input(h)), None)
    if corrected:
        c = letter[corrected]
        for i in range(2, last + 1):
            ws[f"{c}{i}"].fill = _fill(SHEET_INPUT)
        dv = DataValidation(allow_blank=True, showInputMessage=True, promptTitle='Type the right mix',
                            prompt='Sounds joined by " + " (e.g. Howl + Bark + Group), or "ok" if the best guess is '
                                   'right. Leave blank if you did not listen.')
        ws.add_data_validation(dv)
        dv.add(f"{c}2:{c}{last}")
        note = Comment('\n'.join(how_to_correct.iloc[:, 0].astype(str)), 'YipNET')
        note.width, note.height = 420, 260
        ws[f"{c}1"].comment = note

    #Warning cells that are not empty: bold text only
    rng = lambda h: f"{letter[h]}2:{letter[h]}{last}"
    if 'Warning' in letter:
        c = letter['Warning']
        ws.conditional_formatting.add(rng('Warning'), FormulaRule(formula=[f'LEN(${c}2)>0'], font=Font(bold=True)))
    #very light banding that follows the visible rows when filtered (Corrected tag keeps its yellow)
    ws.conditional_formatting.add(' '.join(rng(h) for h in headers if not _is_input(h)),
                                  FormulaRule(formula=['MOD(SUBTOTAL(3,$A$2:$A2),2)=0'], fill=_fill(SHEET_BAND)))

    ws.freeze_panes = 'C2'
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{last}"
    _print_setup(ws, title_rows='1:1')
    return letter


#--- the Summary sheet: plain tables of live formulas, in two columns so the page fits on one landscape sheet

def _hm(hours_ref):
    """Excel text for hours as '6 h 42 min'."""
    minutes = f"INT(ROUND({hours_ref}*60,6))"
    return f'INT({minutes}/60)&" h "&MOD({minutes},60)&" min"'


class _Tables:
    def __init__(self, ws):
        self.ws, self.where = ws, {}

    def _resolve(self, value, prev):
        if not isinstance(value, str):
            return value
        value = value.replace('{prev}', prev or '')
        return re.sub(r'\{key:([^}]+)\}', lambda m: self.where[m.group(1)], value)

    def section(self, top, left, title, rows, note, width=5):
        """A bold heading, label | value rows (with optional green header rows), and a one-line note."""
        ws, r = self.ws, top
        cell = ws.cell(r, left, title)
        cell.font = Font(name='Calibri', size=12, bold=True, color=SHEET_GREEN)
        r += 1
        for row in rows:
            if row and row[0] == '__header__':
                for j in range(width):
                    cell = ws.cell(r, left + j, row[1][j] if j < len(row[1]) else None)
                    _header_cell(cell, align='left' if j == 0 else 'right', wrap=False)
                r += 1
                continue
            label = ws.cell(r, left, row[0])
            label.font, label.border = _FONT, _BOX
            label.alignment = Alignment(vertical='center')
            for j in range(1, width):
                cell = ws.cell(r, left + j)
                cell.border, cell.font = _BOX, _FONT
                item = row[j] if j < len(row) else None
                if item is None:
                    continue
                cell.value = self._resolve(item[0], ws.cell(r, left + j - 1).coordinate)
                cell.number_format = item[1]
                cell.alignment = Alignment(horizontal='right', vertical='center')
                if len(item) > 2:
                    self.where[item[2]] = cell.coordinate
            r += 1
        #the note, one short line per row (no wrapping, so rows shared with the section beside it keep their height)
        for line in [note] if isinstance(note, str) else note:
            cell = ws.cell(r, left, self._resolve(line, None))
            cell.font = Font(name='Calibri', size=10, italic=True, color=SHEET_NOTE)
            r += 1
        return r


def _criteria(rng, lo, hi):
    """COUNTIFS-style criteria pairs for lo <= value < hi."""
    parts = ([f'{rng},">={lo}"'] if lo is not None else []) + ([f'{rng},"<{hi}"'] if hi is not None else [])
    return ','.join(parts)


def _labels_check(out, letter, R):
    """(range, right, labeled, wrong criteria, first cell) for the label comparison column, or None without labels."""
    has = 'Labels in the spreadsheet' in letter and out['Labels in the spreadsheet'].fillna('').astype(str).str.len().gt(0).any()
    if has and 'Best guess right?' in letter:
        return R('Best guess right?'), '"yes"', '"?*"', '"no"', 'Best guess right?'
    if has and 'Match with labels (%)' in letter:
        return R('Match with labels (%)'), '100', '">=0"', '"<100"', 'Match with labels (%)'
    return None


def _style_summary(ws, out, file, letter, title='Batch summary'):
    last = max(len(out) + 1, 2)
    R = lambda h: f"'{DATA_SHEET}'!${letter[h]}$2:${letter[h]}${last}"
    first = lambda h: f"'{DATA_SHEET}'!${letter[h]}$2"
    F, CL, G, CF, W = R('File'), R('Clip length (s)'), R('Best guess'), R('Confidence (%)'), R('Warning')
    CT = R(next(h for h in out.columns if _is_input(h)))
    check = _labels_check(out, letter, R)
    if check:
        RC, right, labeled, wrong, rc_header = check
        RC1 = first(rc_header)

    for col, w in zip('ABCDEFGHIJKL', [2, 36, 16, 16, 16, 20, 3, 36, 16, 16, 16, 20]):
        ws.column_dimensions[col].width = w
    ws.sheet_view.showGridLines = False
    ws['B1'] = title
    ws['B1'].font = Font(name='Calibri', size=16, bold=True, color=SHEET_GREEN)
    ws['B2'] = f"File: {file.replace(chr(92), '/').split('/')[-1]}"
    ws['B3'] = f"Created: {datetime.datetime.now():%Y-%m-%d %H:%M}"
    ws['H2'] = 'Numbers are live formulas on the "Best guesses" sheet: they update as reviewers fill in Corrected tag.'
    ws['H3'] = 'Confidence is the calibrated chance that the whole best guess is right.'
    for c in ('B2', 'B3', 'H2', 'H3'):
        ws[c].font = Font(name='Calibri', size=10, color=SHEET_NOTE)

    t = _Tables(ws)
    pct = '={prev}/{key:overview.clips}'
    band_crit = [_criteria(CF, lo, hi) for lo, hi, _ in SHEET_BANDS]

    #1. Batch overview
    overview = [
        ['Clips', (f'=COUNTA({F})', F_COUNT, 'overview.clips')],
        ['Clips with a warning', (f'=COUNTIF({W},"?*")', F_COUNT, 'overview.warn'), (pct, F_PCT, 'overview.warn_pct')],
        ['Clips with no coyote heard', (f'=COUNTIF({W},"no coyote heard")', F_COUNT, 'overview.nocoyote'),
         (pct, F_PCT, 'overview.nocoyote_pct')],
        ['__header__', ['Confidence', 'Mean', 'Median', 'Min', 'Max']],
        ['All clips', (f'=AVERAGE({CF})', F_CONF, 'overview.conf_mean'), (f'=MEDIAN({CF})', F_CONF, 'overview.conf_median'),
         (f'=MIN({CF})', F_CONF, 'overview.conf_min'), (f'=MAX({CF})', F_CONF, 'overview.conf_max')],
        ['Clips at 80%+ confidence', (f'=COUNTIF({CF},">=80")', F_COUNT, 'overview.conf80'), (pct, F_PCT, 'overview.conf80_pct')],
        ['Clips below 50% confidence', (f'=COUNTIF({CF},"<50")', F_COUNT, 'overview.conf50'), (pct, F_PCT, 'overview.conf50_pct')],
    ]

    #2. Clip length and listening time
    listening = [
        ['__header__', ['', 'Hours', 'Time', 'Seconds']],
        ['Total clip length', (f'=SUM({CL})/3600', F_HOURS, 'time.total_h'), ('=' + _hm('{prev}'), '@', 'time.total_hm'),
         (f'=SUM({CL})', '#,##0.0" s"', 'time.total_s')],
        ['__header__', ['Clip length', 'Mean', 'Median', 'Min', 'Max']],
        ['Seconds', (f'=AVERAGE({CL})', F_SEC, 'time.mean'), (f'=MEDIAN({CL})', F_SEC, 'time.median'),
         (f'=MIN({CL})', F_SEC, 'time.min'), (f'=MAX({CL})', F_SEC, 'time.max')],
        ['__header__', ['To check', 'Clips', 'Time', '% of time'] + (['Wrong guesses found'] if check else [])],
    ]
    for share in (20, 50):
        n_key, h_key = f'time.top{share}.n', f'time.top{share}.h'
        #the listening time of the top rows (the sheet is sorted least confident first)
        hours =f'SUM({first("Clip length (s)")}:INDEX({CL},{{key:{n_key}}}))/3600'
        row = [f'Least confident {share}% of clips',
               (f'=MAX(1,ROUND({share / 100}*{{key:overview.clips}},0))', F_COUNT, n_key),
               ('=' + _hm(hours), '@', f'time.top{share}.hm'),
               (f'={hours}/{{key:time.total_h}}', F_PCT, f'time.top{share}.pct')]
        if check:
            row.append((f'=IFERROR(COUNTIF({RC1}:INDEX({RC},{{key:{n_key}}}),{wrong})/COUNTIF({RC},{wrong}),"-")',
                        F_PCT, f'time.top{share}.found'))
        listening.append(row)
    row = ['All clips', ('={key:overview.clips}', F_COUNT, 'time.all.n'), ('=' + _hm('{key:time.total_h}'), '@', 'time.all.hm'),
           ('=IF({key:time.total_h}>0,1,"-")', F_PCT, 'time.all.pct')]
    if check:
        row.append((f'=IF(COUNTIF({RC},{wrong})>0,1,"-")', F_PCT, 'time.all.found'))
    listening.append(row)
    listening.append(['Clips with a warning', (f'=COUNTIF({W},"?*")', F_COUNT, 'time.warn.n'),
                      ('=' + _hm(f'SUMIF({W},"?*",{CL})/3600'), '@', 'time.warn.hm'),
                      (f'=SUMIF({W},"?*",{CL})/3600/{{key:time.total_h}}', F_PCT, 'time.warn.pct')])
    listening_note = ['The sheet is sorted least confident first: checking its top rows covers the least confident clips.']
    if check:
        listening_note.append('"Wrong guesses found": share of all clips whose best guess differs from the labels.')

    #3. Confidence bands
    bands = [['__header__', ['Confidence band', 'Clips', '% of clips', 'Mean confidence', 'Total length']]]
    for i, ((lo, hi, name), crit) in enumerate(zip(SHEET_BANDS, band_crit)):
        bands.append([name, (f'=COUNTIFS({crit})', F_COUNT, f'bands.{i}.n'), (pct, F_PCT, f'bands.{i}.pct'),
                      (f'=IFERROR(AVERAGEIFS({CF},{crit}),"-")', F_CONF, f'bands.{i}.conf'),
                      ('=' + _hm(f'SUMIFS({CL},{crit})/3600'), '@', f'bands.{i}.hm')])
    bands.append(['All clips', ('={key:overview.clips}', F_COUNT, 'bands.all.n'), (pct, F_PCT, 'bands.all.pct'),
                  (f'=AVERAGE({CF})', F_CONF, 'bands.all.conf'), ('=' + _hm('{key:time.total_h}'), '@', 'bands.all.hm')])
    if check:
        #"found every labeled sound": the best guess misses none of the labels' sounds (it may add others)
        MS = R('Model missed') if 'Model missed' in letter else None
        bands.append(['__header__', ['Confidence band', 'Clips with labels', '% really right', 'Mean confidence']
                      + (['Found all labeled'] if MS else [])])
        for i, ((lo, hi, name), crit) in enumerate(zip(SHEET_BANDS, band_crit)):
            row = [name, (f'=COUNTIFS({crit},{RC},{labeled})', F_COUNT, f'bands.{i}.labeled'),
                   (f'=IFERROR(COUNTIFS({crit},{RC},{right})/{{key:bands.{i}.labeled}},"-")', F_PCT, f'bands.{i}.right'),
                   (f'={{key:bands.{i}.conf}}', F_CONF, f'bands.{i}.conf2')]
            if MS:
                row.append((f'=IFERROR(COUNTIFS({crit},{RC},{labeled},{MS},"")/{{key:bands.{i}.labeled}},"-")', F_PCT,
                            f'bands.{i}.found'))
            bands.append(row)
        row = ['All clips', (f'=COUNTIF({RC},{labeled})', F_COUNT, 'bands.all.labeled'),
               (f'=IFERROR(COUNTIF({RC},{right})/{{key:bands.all.labeled}},"-")', F_PCT, 'bands.all.right'),
               ('={key:bands.all.conf}', F_CONF, 'bands.all.conf2')]
        if MS:
            row.append((f'=IFERROR(COUNTIFS({RC},{labeled},{MS},"")/{{key:bands.all.labeled}},"-")', F_PCT, 'bands.all.found'))
        bands.append(row)
    bands_note = (['An honest model is right about as often as it says: "% really right" (best guess = labels exactly) should',
                   'be close to the mean confidence. "Found all labeled": the guess has every labeled sound (it may add more).']
                  if check else 'How many clips fall in each confidence band, and how long they take to hear.')

    #4. Best guesses, most common first
    counts = out['Best guess'].value_counts()
    mixes = sorted(counts.index, key=lambda m: (-counts[m], m))
    guesses = [['__header__', ['Best guess', 'Clips', '% of clips', 'Mean confidence']]]
    for i, mix in enumerate(mixes):
        guesses.append([mix, (f'=COUNTIF({G},"{mix}")', F_COUNT, f'mix.{i}.n'), (pct, F_PCT, f'mix.{i}.pct'),
                        (f'=AVERAGEIF({G},"{mix}",{CF})', F_CONF, f'mix.{i}.conf')])

    #5. Each sound
    sounds = [h[:-len(' chance (%)')] for h in out.columns if h.endswith(' chance (%)')]
    each = [['__header__', ['Sound', 'Mean chance', 'Clips at 50%+', 'In best guess']]]
    for s in sounds:
        col = R(f'{s} chance (%)')
        each.append([s, (f'=AVERAGE({col})', F_CONF, f'sound.{s}.mean'), (f'=COUNTIF({col},">=50")', F_COUNT, f'sound.{s}.n50'),
                     (f'=SUMPRODUCT(--ISNUMBER(SEARCH(" + {s} + "," + "&{G}&" + ")))', F_COUNT, f'sound.{s}.guess')])

    #6. Longest howl
    HS, HP = R('Longest howl (s)'), R('Longest howl pitch (Hz)')
    howl = [
        ['Clips whose best guess has a howl', (f'=SUMPRODUCT(--ISNUMBER(SEARCH(" + Howl + "," + "&{G}&" + ")))', F_COUNT, 'howl.guessed')],
        ['Howls measured', (f'=COUNT({HS})', F_COUNT, 'howl.measured')],
        ['Howl guessed, none measured', (f'=COUNTIF({HS},"Howl guessed*")', F_COUNT, 'howl.none')],
        ['"No howl detected"', (f'=COUNTIF({HS},"No howl detected")', F_COUNT, 'howl.nohowl')],
        ['__header__', ['', 'Mean', 'Median', 'Min', 'Max']],
        ['Longest howl', (f'=IFERROR(AVERAGE({HS}),"-")', F_SEC, 'howl.s.mean'), (f'=IFERROR(MEDIAN({HS}),"-")', F_SEC, 'howl.s.median'),
         (f'=IF(COUNT({HS}),MIN({HS}),"-")', F_SEC, 'howl.s.min'), (f'=IF(COUNT({HS}),MAX({HS}),"-")', F_SEC, 'howl.s.max')],
        ['Pitch', (f'=IFERROR(AVERAGE({HP}),"-")', F_HZ, 'howl.hz.mean'), (f'=IFERROR(MEDIAN({HP}),"-")', F_HZ, 'howl.hz.median'),
         (f'=IF(COUNT({HP}),MIN({HP}),"-")', F_HZ, 'howl.hz.min'), (f'=IF(COUNT({HP}),MAX({HP}),"-")', F_HZ, 'howl.hz.max')],
        ['__header__', ['Longest howl at least', 'Clips']],
    ] + [[f'{s} s', (f'=COUNTIF({HS},">={s}")', F_COUNT, f'howl.atleast{s}')] for s in (5, 10, 20)]

    #7. Warnings
    warn = [['__header__', ['Warning', 'Clips', 'Total length']]]
    for key, name, crit in [('nocoyote', 'no coyote heard', '"no coyote heard"'),
                            ('unfamiliar', 'whole clip unfamiliar', '"whole clip unfamiliar"'),
                            ('unidentified', 'unidentified chunk', '"unidentified*"'), ('any', 'Any warning', '"?*"')]:
        warn.append([name, (f'=COUNTIF({W},{crit})', F_COUNT, f'warn.{key}.n'),
                     ('=' + _hm(f'SUMIF({W},{crit},{CL})/3600'), '@', f'warn.{key}.hm')])

    #8. Review progress
    review = [
        ['Clips checked', (f'=COUNTIF({CT},"?*")', F_COUNT, 'review.checked'), (pct, F_PCT, 'review.checked_pct')],
        ['Listening time checked', ('=' + _hm(f'SUMIF({CT},"?*",{CL})/3600'), '@', 'review.hm')],
        ['Marked "ok"', (f'=COUNTIF({CT},"ok")', F_COUNT, 'review.ok')],
        ['Typed the same mix as the best guess', (f'=SUMPRODUCT((LEN({CT})>0)*({CT}={G}))', F_COUNT, 'review.same')],
        ['Corrected to a different mix', ('={key:review.checked}-{key:review.ok}-{key:review.same}', F_COUNT, 'review.changed')],
        ['__header__', ['Confidence band', 'Checked', 'Clips in band', '% checked']],
    ]
    for i, ((lo, hi, name), crit) in enumerate(zip(SHEET_BANDS, band_crit)):
        review.append([name, (f'=COUNTIFS({crit},{CT},"?*")', F_COUNT, f'review.band{i}.n'),
                       (f'={{key:bands.{i}.n}}', F_COUNT, f'review.band{i}.of'),
                       (f'=IFERROR({{key:review.band{i}.n}}/{{key:bands.{i}.n}},"-")', F_PCT, f'review.band{i}.pct')])

    #two columns of sections (left B-F, right H-L), each pair starting below the taller of the previous pair
    grid = [(('1. Batch overview', overview, 'How many clips, how many need a listen, and how sure the model is.'),
             ('2. Clip length and listening time', listening, listening_note)),
            (('3. Confidence bands', bands, bands_note),
             ('4. Best guesses', guesses, 'Each distinct mix of sounds the model guessed, most common first.')),
            (('5. Each sound', each, 'Mean chance over all clips; clips at 50% or more; best guesses that include it.'),
             ('6. Longest howl', howl, 'Clips whose best guess has a howl; measured with a pitch tracker, so approximate.')),
            (('7. Warnings', warn, 'Clips that failed a review check; these always need a listen.'),
             ('8. Review progress', review, 'Updates as reviewers fill in Corrected tag ("ok" or the same mix = confirmed).'))]
    top = 5
    for left_section, right_section in grid:
        top = max(t.section(top, 2, *left_section), t.section(top, 8, *right_section)) + 1
    _print_setup(ws, one_page=True)
    ws.print_area = f'A1:L{top}'
    return t.where


#--- the Column guide sheet

def _column_meaning(h, how_to_correct, labels_source='labels_train.xlsx'):
    """(what it means, units, notes) for a Best guesses column."""
    guide = {
        'File': ('The recording\'s file name.', '', 'Find and play the clip in the audio folder.'),
        'Clip length (s)': ('The recording\'s duration, read from the WAV file\'s header with the soundfile library '
                            '(frames / sample rate) and rounded to 0.1 s.', 'seconds',
                            'The Summary adds these up into listening time.'),
        'Best guess': ('The model\'s most likely mix of sounds (at least one call; Group whenever Chorus).', '',
                       'If it is wrong, type the right mix in Corrected tag.'),
        'Confidence (%)': ('The calibrated chance that the whole best guess is exactly right: of clips stated at 80%, '
                           'about 80% should be right.', '%', 'The sheet is sorted least confident first: start at the top.'),
        'Least sure part': ('The one sound whose present/absent call the model is least sure of, with the chance that '
                            'call is right.', '', 'Listen for this sound first.'),
        'Longest howl (s)': ('The longest continuous howl inside the clip\'s Howl stretches (the 30% sound times), measured '
                             'with the pyin pitch tracker on noise-cleaned audio (dips up to 0.25 s bridged). "No howl detected" when '
                             'the best guess has no howl; "Howl guessed, no continuous howl measured" when the tracker found no tone; '
                             '"Howl guessed, no continuous howl in the howl times" when the tones it found were all outside the Howl '
                             'stretches.', 'seconds', 'Approximate.'),
        'Longest howl starts': ('Where that longest howl starts in the recording.', 'm:ss', 'Approximate.'),
        'Longest howl pitch (Hz)': ('The median pitch (fundamental frequency) over that howl; never below 100 Hz.', 'Hz',
                                    'Approximate.'),
        'Warning': ('Review checks: "no coyote heard" (BirdNET\'s Coyote score never reaches 0.005), "whole clip '
                    'unfamiliar" (unlike every training clip), "unidentified at ..." (a 3-second chunk unlike every '
                    'training chunk).', '', 'Always listen to these clips; the best guess may not apply.'),
        'Labels in the spreadsheet': (f'The sounds in the team\'s original labels ({labels_source}), written '
                                      'like the best guess.', '',
                                      'Only on sheets of labeled clips; blank for recordings not in that spreadsheet.'),
        'Best guess right?': ('"yes" when the best guess exactly matches the labels.', '',
                              'Only on sheets of labeled clips.'),
        'Match with labels (%)': ('The sounds both the best guess and the labels name, divided by the sounds either names '
                                  '(100% = exact match).', '%', 'Only on sheets of labeled clips.'),
        'Model added': ('Sounds in the best guess that the labels do not have.', '',
                        'Only on sheets of labeled clips. The model may have caught something untagged, or be wrong.'),
        'Model missed': ('Sounds in the labels that the best guess does not have.', '',
                         'Only on sheets of labeled clips. Often short calls the model hears less well.'),
    }
    if h in guide:
        return guide[h]
    if h.endswith(' chance (%)'):
        sound = h[:-len(' chance (%)')]
        what = 'more than one coyote is calling' if sound == 'Group' else f'a {sound} is present anywhere in the clip'
        return f'The calibrated chance that {what}.', '%', 'Sounds in the best guess are usually above 50%.'
    if h.startswith('Sound times') and '30%' in h:
        return ('For each sound in the best guess, the 3-second chunks where it is at least 30% likely, joined into '
                'stretches; "(strongest)" when no chunk reaches 30%. A wider net than the 50% column: it also shows '
                'quieter or less certain stretches.', 'm:ss-m:ss',
                'Approximate: the model was never taught times, and times are whole 3-second chunks.')
    if h.startswith('Sound times'):
        return ('For each sound in the best guess, the 3-second chunks where the model is at least 50% sure it is there, '
                'joined into stretches; "(strongest)" marks a sound with no chunk at 50% (its most likely chunk is given).',
                'm:ss-m:ss', 'Approximate: the model was never taught times, and times are whole 3-second chunks.')
    if _is_input(h):
        return (' '.join(how_to_correct.iloc[:, 0].astype(str)), 'text',
                f'The only column to type in (light yellow). Full header: {h}')
    return '', '', ''


def _style_column_guide(ws, out, how_to_correct, labels_source='labels_train.xlsx'):
    ws.append(['Column', 'What it means', 'Units', 'Notes'])
    for h in out.columns:
        ws.append([h, *_column_meaning(h, how_to_correct, labels_source)])
    for col, w in zip('ABCD', [30, 80, 11, 55]):
        ws.column_dimensions[col].width = w
    for cell in ws[1]:
        _header_cell(cell, align='left')
    ws.row_dimensions[1].height = 22
    for r, row in enumerate(ws.iter_rows(min_row=2), start=2):
        for cell in row:
            cell.font, cell.border = _FONT, _BOX
            cell.alignment = Alignment(wrap_text=True, vertical='top')
            if r % 2 == 1:
                cell.fill = _fill(SHEET_BAND)
        row[0].font = Font(name='Calibri', size=11, bold=True)
        lines = max(math.ceil(len(str(row[0].value or '')) / 30), math.ceil(len(str(row[1].value or '')) / 85),
                    math.ceil(len(str(row[3].value or '')) / 58), 1)
        ws.row_dimensions[r].height = 15 * min(lines, 10) + 3
    ws.freeze_panes = 'B2'
    _print_setup(ws, title_rows='1:1')


def _style_how_to_correct(ws):
    ws.column_dimensions['A'].width = 110
    for r, row in enumerate(ws.iter_rows(), start=1):
        cell = row[0]
        if r == 1:
            _header_cell(cell, align='left')
        else:
            cell.font, cell.border = _FONT, _BOX
            cell.alignment = Alignment(wrap_text=True, vertical='top')
            ws.row_dimensions[r].height = 15 * max(1, math.ceil(len(str(cell.value or '')) / 115)) + 3
    _print_setup(ws)


def write_best_guess_workbook(out, file, how_to_correct, title='Batch summary',
                              labels_source='labels_train.xlsx'):
    """Write `out` to a tidy workbook (Summary, Best guesses, How to correct, Column guide); returns the Summary's cells.
    `title` heads the Summary; `labels_source` is the labels spreadsheet the Column guide names."""
    with pd.ExcelWriter(file, engine='openpyxl') as writer:
        out.to_excel(writer, sheet_name=DATA_SHEET, index=False)
        how_to_correct.to_excel(writer, sheet_name='How to correct', index=False)
        wb = writer.book
        letter = _style_best_guesses(writer.sheets[DATA_SHEET], out, how_to_correct)
        where = _style_summary(wb.create_sheet('Summary', 0), out, file, letter, title)
        _style_how_to_correct(writer.sheets['How to correct'])
        _style_column_guide(wb.create_sheet('Column guide'), out, how_to_correct, labels_source)
        wb.active = 0
        for ws in wb.worksheets:
            ws.sheet_view.tabSelected = ws.title == 'Summary'
        wb.calculation.fullCalcOnLoad = True
    return where

In [ ]:
#best_guess_sheet writes one row per clip: File, Clip length (s) (the WAV's duration), Best guess,
#Confidence (%), Least sure part (the sound whose present/absent call is least sure, with that call's chance of being
#right), each sound's chance (%), the sound times, the longest howl (length, start and pitch), Warning (the review
#checks), a blank Corrected tag column for the reviewers (its title gives the format), and, for clips in the training
#spreadsheet, their labels and how far the best guess matches them (Match with labels (%) = the sounds both name over
#the sounds either names; Model added / Model missed = the sounds only the model or only the labels name). Least
#confident first by default (`order`). When `file` is given it writes the spreadsheet (with a "How to correct" sheet) and
#the clips' BirdNET numbers next to it (`<file>_birdnet.npz`), so corrections can retrain the model without running
#BirdNET again; it never overwrites an existing file. The training clips' BirdNET output is read from the cache, and
#`reuse` (an earlier best-guess sheet) supplies the longest-howl columns already measured, so a rebuild is fast.
#`files` limits the sheet to those file names in `folder`; `labels` ({file name: mix}) gives labels from another
#spreadsheet (the test data, Section 9) instead of the training labels; `title` heads the Summary and
#`labels_source` is the labels spreadsheet the Column guide names
ORDERS20 = ['least confident first', 'most confident first']
STRICT_TIMES_COL = 'Sound times (approx.; chunks at least 50% sure)'
LOOSE_TIMES_COL = 'Sound times (approx.; chunks at least 30% sure)'
CORRECTED_COL = ('Corrected tag (format: sounds joined by " + ", e.g. Howl + Bark + Group; names: '
                 + ', '.join(TAGS) + '; or "ok" if the best guess is right)')
HOW_TO_CORRECT = pd.DataFrame({'How to correct': [
    'Listen to each clip, starting at the top (least confident first).',
    'If the best guess is wrong, type the right mix in the "Corrected tag" column, using the names Howl, Bark, Yip-Yap, Chorus, Whine, '
    'Growl and Group joined by " + " (for example: Howl + Bark + Group). Group means more than one coyote; Chorus always counts as Group.',
    'If the best guess is right, type it again (or "ok") so the clip counts as checked.',
    'Leave "Corrected tag" blank for clips you did not check.',
    'Save the file, then run learn_from_corrections("this file.xlsx") in the notebook.']})


def compare_with_labels(label, guess):
    """(Match with labels %, Model added, Model missed) for one clip: the sounds both name over the sounds either names."""
    if not label:
        return '', '', ''
    human, model_guess = set(label.split(' + ')), set(guess.split(' + '))
    in_order = lambda sounds: ', '.join(t for t in TAGS if t in sounds)
    return (round(100 * len(human & model_guess) / len(human | model_guess)),
            in_order(model_guess - human), in_order(human - model_guess))


def best_guess_sheet(folder, file=None, order='least confident first', model=None, reuse=None, files=None, labels=None,
                     title='Batch summary', labels_source='labels_train.xlsx'):
    """The final tool: every WAV file in `folder` with its best guess of the sounds and an honest confidence."""
    assert order in ORDERS20, f"order must be one of {ORDERS20}"
    model = model or final20
    names = files if files is not None else [f for f in os.listdir(folder) if f.lower().endswith('.wav')]
    paths = sorted(os.path.join(folder, f) for f in names)
    missing = [p for p in paths if not os.path.exists(p)]
    assert not missing, f"not found: {missing[:5]}"
    #BirdNET output: the training clips' is already cached in `clips`, so only other recordings go through BirdNET
    cached = {os.path.abspath(p): k for k, p in enumerate(dat.audio_path)}
    todo = [p for p in paths if os.path.abspath(p) not in cached]
    fresh = dict(zip(todo, birdnet_batch(todo))) if todo else {}
    new_clips = [clips[cached[os.path.abspath(p)]] if os.path.abspath(p) in cached else fresh[p] for p in paths]
    summary = coyote_moments(new_clips)
    P = model['chance'](summary)
    mix, raw = best_mix(P, model.get('cutoffs'))
    confidence = model['recal'](raw)
    sure = np.where(mix == 1, P, 1 - P)                              # each sound's call: chance it is right
    weakest = sure.argmin(1)
    fq = final_checks
    times = [sound_times(emb, scores['coyote'], m, model) for (emb, scores), m in zip(new_clips, mix)]
    loose_times = [sound_times(emb, scores['coyote'], m, model, chance=LOOSE_TIME_CHANCE)
                   for (emb, scores), m in zip(new_clips, mix)]
    #longest howl: reuse an earlier sheet's measurement of the same file when there is one (pyin is slow)
    earlier = {}
    if reuse:
        old = pd.read_excel(reuse, sheet_name='Best guesses')
        earlier = {r['File']: tuple('' if pd.isna(v) else v for v in
                                    (r['Longest howl (s)'], r['Longest howl starts'], r['Longest howl pitch (Hz)']))
                   for _, r in old.iterrows() if r['Longest howl (s)'] != 'No howl detected'}
    #only tones inside the clip's Howl stretches (30% sound times) count. An earlier sheet's measurement is reused when it is
    #sure to lie in them (its start is rounded down to the second, so 1 s of margin), or when it found no tone at all
    spans = [howl_spans(emb, scores['coyote'], model) for emb, scores in new_clips]

    def reusable(h, sp):
        if h is None or h[0] == 'Howl guessed, no continuous howl in the howl times':
            return False
        if h[0] == 'Howl guessed, no continuous howl measured':
            return True
        try:
            minutes, seconds = str(h[1]).split(':')
            start, length = int(minutes) * 60 + int(seconds), float(h[0])
        except ValueError:
            return False
        return any(start + 1 <= end and start + length - 0.1 > begin for begin, end in sp)
    howls = [howl_columns(False, p) if m[TAGS.index('Howl')] != 1
             else earlier[os.path.basename(p)] if reusable(earlier.get(os.path.basename(p)), sp) else howl_columns(True, p, sp)
             for m, p, sp in zip(mix, paths, spans)]
    #the training spreadsheet's labels, in the same form as the best guess, so the two can be compared
    known = labels if labels is not None else {os.path.basename(p): mix_name(truth20[k]) for k, p in enumerate(dat.audio_path)}
    labels = [known.get(os.path.basename(p), '') for p in paths]
    compared = [compare_with_labels(lab, mix_name(m)) for lab, m in zip(labels, mix)]
    unfamiliar = fq['clip_nearest'].kneighbors(fq['novelty_scaler'].transform(summary))[0][:, 0] > fq['clip_cutoff']
    warnings = []
    for (emb, scores), odd in zip(new_clips, unfamiliar):
        odd_seconds = [seconds_label(k) for k in np.flatnonzero(
            fq['chunk_nearest'].kneighbors(emb.astype('float32'))[0][:, 0] > fq['chunk_cutoff'])]
        warnings.append('no coyote heard' if scores['coyote'].max() < NO_COYOTE_BELOW else 'whole clip unfamiliar' if odd
                        else f"unidentified at {', '.join(odd_seconds)}" if odd_seconds else '')
    out = pd.DataFrame({'File': [os.path.basename(p) for p in paths],
                        'Clip length (s)': [round(sf.info(p).duration, 1) for p in paths],
                        'Best guess': [mix_name(m) for m in mix],
                        'Confidence (%)': np.round(100 * confidence),
                        'Least sure part': [f"{TAGS[w]} {'present' if m[w] else 'absent'} ({s[w]:.0%} sure)"
                                            for w, m, s in zip(weakest, mix, sure)],
                        **{f"{t} chance (%)": np.round(100 * P[:, j]) for j, t in enumerate(TAGS)},
                        STRICT_TIMES_COL: times, LOOSE_TIMES_COL: loose_times,
                        'Longest howl (s)': [h[0] for h in howls], 'Longest howl starts': [h[1] for h in howls],
                        'Longest howl pitch (Hz)': [h[2] for h in howls],
                        'Warning': warnings, CORRECTED_COL: '',
                        'Labels in the spreadsheet': labels,
                        'Match with labels (%)': [c[0] for c in compared],
                        'Model added': [c[1] for c in compared], 'Model missed': [c[2] for c in compared]})
    keep = np.argsort(confidence if order == 'least confident first' else -confidence, kind='stable')
    out, summary = out.iloc[keep].reset_index(drop=True), summary[keep]
    if file:
        features_file = os.path.splitext(file)[0] + '_birdnet.npz'
        for f in (file, features_file):
            if os.path.exists(f):
                raise FileExistsError(f"{f} already exists (it may hold corrections); choose another name")
        write_best_guess_workbook(out, file, HOW_TO_CORRECT, title, labels_source)
        np.savez_compressed(features_file, files=out['File'].values, summary=summary)
        print(f"Saved {file} and {features_file}")
    if any(labels):
        match = [c[0] for c in compared if c[0] != '']
        print(f"Best guess vs the spreadsheet's labels: exact for {match.count(100)} of {len(match)} clips, "
              f"average match {np.mean(match):.0f}%" + ('' if labels is not None else ' (training clips: optimistic)'))
    return out


print("The final tool on the two made-up sounds. For a folder of recordings: "
      "best_guess_sheet('my_folder', file='best_guesses.xlsx'), least confident first (order='most confident first' to flip)")
display(best_guess_sheet('demo_sounds'))

**Next cell:** defines `learn_from_corrections` (it runs once the reviewers have filled in "Corrected tag").

In [ ]:
from sklearn.model_selection import GroupKFold

#learn_from_corrections(*files) reads the reviewers' Corrected tag
#column from one or more best-guess sheets. First it reports how often the best guesses were really right on the checked
#clips, by stated confidence: the tool's real accuracy on new recordings. Then it retrains the final best-guess models on
#the 845 clips plus every checked clip (the final models' method; a checked clip's group is its recorder and night from the
#file name), and the retrained models are used by best_guess_sheet from then on. Give it every corrected sheet so far, so
#nothing earlier is lost. The checked clips are not used to judge the retrained models: the next batch's check does that
#("ok" = the best guess was right)
GROUP_OF_FILE = lambda f: (lambda p: f"{p[0]} {p[1][:4]}-{p[1][4:6]}-{p[1][6:8]}" if len(p) > 1 and len(p[1]) >= 8 else f)(
    os.path.splitext(os.path.basename(str(f)))[0].replace('_', '-').split('-'))


def parse_mix(text, guess):
    """A Corrected tag as a 0/1 row over TAGS ("ok" = the best guess); None when blank."""
    text = str(text).strip()
    if text in ('', 'nan'):
        return None
    if text.lower() == 'ok':
        text = guess
    names = {t.lower().replace('-', '').replace(' ', ''): t for t in TAGS}
    row = np.zeros(len(TAGS), dtype=int)
    for part in text.split('+'):
        key = part.strip().lower().replace('-', '').replace(' ', '')
        if key not in names:
            raise ValueError(f"'{part.strip()}' in '{text}' is not one of {TAGS}")
        row[TAGS.index(names[key])] = 1
    if row[TAGS.index('Chorus')]:
        row[TAGS.index('Group')] = 1                                  # Chorus always counts as Group
        if CHORUS_INCLUDES_HOWL:
            row[TAGS.index('Howl')] = 1                               # and as Howl, as in the training labels
    return row


corrections_log20 = []


def learn_from_corrections(*files):
    """Score the best guesses against the reviewers' corrections, then retrain on the 845 clips plus the checked clips."""
    global final20
    checked, X_new, groups_new = [], [], []
    for file in files:
        sheet = pd.read_excel(file, sheet_name='Best guesses')
        #columns found by how their headers start, so a longer or shorter header still works (extra columns are ignored)
        column = lambda start: next(c for c in sheet.columns if str(c).startswith(start))
        file_col, guess_col, conf_col, corrected_col = map(column, ['File', 'Best guess', 'Confidence', 'Corrected tag'])
        stored = np.load(os.path.splitext(file)[0] + '_birdnet.npz', allow_pickle=True)
        summary_of = dict(zip(stored['files'], stored['summary']))
        for _, row in sheet.iterrows():
            truth = parse_mix(row[corrected_col], row[guess_col])
            if truth is not None:
                checked.append({'File': row[file_col], 'best guess': row[guess_col], 'confidence': row[conf_col] / 100,
                                'corrected': mix_name(truth), 'truth': truth})
                X_new.append(summary_of[row[file_col]])
                groups_new.append(GROUP_OF_FILE(row[file_col]))
    if not checked:
        print('No Corrected tag filled in yet: nothing to learn.')
        return None
    c = pd.DataFrame(checked)
    c['right'] = c['best guess'] == c['corrected']
    c['band'] = pd.cut(c.confidence, BANDS20, right=False, labels=BAND_NAMES20)
    print(f"{len(c)} checked clips: the best guess was right for {c.right.mean():.1%} (stated confidence {c.confidence.mean():.1%} "
          'on average). By stated confidence:')
    display(c.groupby('band', observed=True).agg(clips=('right', 'size'), stated=('confidence', 'mean'),
                                                 really_right=('right', 'mean')).round(3))

    #retrain on the 845 clips plus the checked clips
    started = time.time()
    X = np.vstack([V_birdnet_moments, np.array(X_new)])
    truth = np.vstack([truth20, np.array(c['truth'].tolist())])
    groups = np.concatenate([np.asarray(FINAL_SPLIT['groups'], dtype=object), np.array(groups_new, dtype=object)])
    rows = np.arange(len(X))
    folds = []
    for rest_idx, held_idx in GroupKFold(n_splits=N_FOLDS).split(rows, groups=groups):
        rest = rows[rest_idx]
        _, stop_idx = next(GroupKFold(n_splits=5).split(rest, groups=groups[rest]))
        folds.append((rest, rows[held_idx], rest[stop_idx]))
    fit_idx, stop_idx = next(GroupKFold(n_splits=5).split(rows, groups=groups))
    final20 = train_best_guess(X, truth, rows[fit_idx], rows[stop_idx], folds)
    corrections_log20.append({'round': len(corrections_log20) + 1, 'checked clips': len(c), 'best guess right': c.right.mean(),
                              'stated confidence': c.confidence.mean(), 'training clips now': len(X)})
    print(f"Retrained on {len(X)} clips ({len(X) - len(dat)} checked clips added; {time.time() - started:.0f}s). "
          'best_guess_sheet now uses the retrained models. Correction rounds so far:')
    display(pd.DataFrame(corrections_log20).set_index('round').round(3))
    return c


print('learn_from_corrections is ready: after the reviewers fill in "Corrected tag", run '
      'learn_from_corrections("best_guesses.xlsx") (list every corrected sheet so far).')

## 9. Test data: the labeled clips of Test #1

Everything above was built on the 845 training clips. This section is the honest check: the **final tool, unchanged**,
tags clips it has never heard, and the team's test labels are used **only to score** it.

**At a glance**

- **Question:** How well does the final tool do on new, labeled recordings, and how does that compare with training?
- **Method:** The final models of Section 8 (trained on the 845 training clips only; the per-sound cutoffs and the
  confidence recalibration were also set on training clips) tag the 181 clips listed in `labels_test1.xlsx`
  (audio in `Test_Audio_Files/`). The test labels are cleaned exactly as the training labels (Section 1) and turned into
  mixes the same way (Section 6, including Chorus counts as Group and as Howl). WAV files in the folder that the test
  spreadsheet does not list are skipped (they are for a later test). No test clip is in the training data.
- **Result (saved run, Nov 15):** On the 181 test clips the best guess is exactly right for **66.9%** (95% range 59.7% to 73.3%), against 57.5% for always guessing Howl, and against **61.9%** on the training clips held out (6 splits): the held-out result is inside the test's range, so the test agrees with it. Found every labeled sound 74.6% (held out 69.2%). The confidence is honest: stated 66.5%, really right 66.9%. As in training, pure howls are right 93% of the time and mixes much less (Howl + Bark 14%, Howl + Group 40%, Howl + Chorus + Group 50%). 179 of the 181 test clips come from recorders also in the training data, so this tests new recordings, not new places (Section 8's recorder-only splits cover that).
- **Output:** a test best-guess sheet, `TEST_best_guesses_YYYY-MM-DD.xlsx` (Summary titled "TEST DATA"; a new name each
  run, never overwritten), and a table comparing training and test.
- **How to read the comparison:** "training, seen" is the final models on the clips they learned from (optimistic);
  "training, held out" is Section 8's 6 splits (each clip scored by models that never saw it), the fair yardstick;
  "test" is the new clips. With 181 test clips, differences of a few points are within chance (the 95% range shows how much).

**Next cell:** reads the test labels and makes the test best-guess sheet (BirdNET and the howl pitch tracker run on the
test clips: a few minutes).

In [ ]:
import datetime

#the test data: the final models (final20, trained on the 845 training clips only) tag the labeled clips of the test
#spreadsheet; the test labels are read only to score the guesses, never to train or tune anything. WAV files in
#TEST_AUDIO_DIR that the test spreadsheet does not list are skipped (they are for a later test)
TEST_LABELS_FILE = 'labels_test1.xlsx'
TEST_AUDIO_DIR = 'Test_Audio_Files'

test = pd.read_excel(TEST_LABELS_FILE)
#the same cleaning as the training labels (Section 1); a column that is empty in the test sheet stays empty
tidy = lambda s: s.astype(object).where(s.isna(), s.astype(str).str.strip())
for column in LABEL_COLUMNS:
    test[column] = tidy(test[column]).map(lambda v: v.title() if isinstance(v, str) else v)
test['ARU or Location'] = tidy(test['ARU or Location'])
test['File'] = test['Original File'].astype(str).str.strip()
has_test_audio = test.File.map(lambda f: os.path.exists(os.path.join(TEST_AUDIO_DIR, f)))
print(f"{has_test_audio.sum()} of {len(test)} test clips found in {TEST_AUDIO_DIR}/")
test = test[has_test_audio].sort_values('File').reset_index(drop=True)
#names outside the seven sounds are left out, as for the training labels (Section 6)
unknown = test[LABEL_COLUMNS].stack()
unknown = unknown[~unknown.isin(CALLS)]
if len(unknown):
    print('Label names that are not one of the seven sounds (left out, as in training):',
          unknown.value_counts().to_dict())
in_training = set(test.File) & {os.path.basename(p) for p in dat.audio_path}
assert not in_training, f"test clips also in the training data: {in_training}"
wavs = [f for f in os.listdir(TEST_AUDIO_DIR) if f.lower().endswith('.wav')]
print(f"{len(set(wavs) - set(test.File))} other WAV files in {TEST_AUDIO_DIR}/ have no test label and are skipped")

#each test clip's true mix, built exactly as for the training clips (Section 6)
test_any_group = test['Group'].notna() | (test[LABEL_COLUMNS] == 'Chorus').any(axis=1)
truth_test = np.array([[(tag == 'Group' and group) or (tag != 'Group' and tag in set(calls)) for tag in TAGS]
                       for calls, group in zip(test[LABEL_COLUMNS].values, test_any_group)], dtype=int)
if CHORUS_INCLUDES_HOWL:
    truth_test[truth_test[:, TAGS.index('Chorus')] == 1, TAGS.index('Howl')] = 1
TEST_LABELS = {f: mix_name(t) for f, t in zip(test.File, truth_test)}
seen_recorder = test['ARU or Location'].isin(set(dat['ARU or Location']))
print(f"Test clips from recorders that are also in the training data: {seen_recorder.sum()} of {len(test)}")
print('\nTest clips by main call type')
print(test['Call Type'].value_counts().to_string())
print('\nTest clips with each sound present')
print(pd.Series(truth_test.sum(0), index=TAGS).to_string())

#a new file name each run, so an existing (possibly corrected) test sheet is never overwritten
stem = f"TEST_best_guesses_{datetime.date.today():%Y-%m-%d}"
TEST_SHEET_FILE, n = f"{stem}.xlsx", 1
while os.path.exists(TEST_SHEET_FILE) or os.path.exists(os.path.splitext(TEST_SHEET_FILE)[0] + '_birdnet.npz'):
    n += 1
    TEST_SHEET_FILE = f"{stem}_{n}.xlsx"
print()
test_sheet = best_guess_sheet(TEST_AUDIO_DIR, file=TEST_SHEET_FILE, files=list(test.File), labels=TEST_LABELS,
                              title='TEST DATA: batch summary', labels_source=TEST_LABELS_FILE)
print('Test spreadsheet saved to:', os.path.abspath(TEST_SHEET_FILE))

**Next cell:** training against test: the whole mix (exactly right, found every labeled sound, how honest the
confidence is, against always guessing Howl) and each sound (labeled clips caught, false alarms), then the test clips
by confidence band and by true mix.

In [ ]:
#three views of the same final tool: training clips seen in training (optimistic), training clips held out (Section 8's
#6 splits, the fair yardstick) and the test clips. The test guesses are recomputed from the BirdNET numbers saved with the
#test sheet, unrounded
stored = np.load(os.path.splitext(TEST_SHEET_FILE)[0] + '_birdnet.npz', allow_pickle=True)
row_of = {f: i for i, f in enumerate(stored['files'])}
P_test = final20['chance'](stored['summary'][[row_of[f] for f in test.File]].astype(float))
mix_test, raw_test = best_mix(P_test, final20['cutoffs'])
conf_test = final20['recal'](raw_test)
on_sheet = test_sheet.set_index('File').loc[test.File]
assert (on_sheet['Best guess'].values == [mix_name(m) for m in mix_test]).all()
assert np.allclose(on_sheet['Confidence (%)'].values, np.round(100 * conf_test))

mix_seen, raw_seen = best_mix(final20['chance'](V_birdnet_moments), final20['cutoffs'])
conf_seen = final20['recal'](raw_seen)
mix_held = np.array([[t in g.split(' + ') for t in TAGS] for g in all20['best guess']], dtype=int)
truth_held = truth20[all20.index]
conf_held = all20.confidence.values
HOWL_ONLY = np.array([t == 'Howl' for t in TAGS], dtype=int)


def wilson(k, n, z=1.96):
    """95% range for a share of k out of n (Wilson score interval)."""
    p, z2 = k / n, z * z
    centre, half = (p + z2 / (2 * n)) / (1 + z2 / n), z * np.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / (1 + z2 / n)
    return centre - half, centre + half


def performance(mix, truth, conf, per_split=1):
    """The comparison rows for one view (per_split: how many times each clip is counted)."""
    right = (mix == truth).all(1)
    both, either = (mix & truth).sum(1), (mix | truth).sum(1)
    howl = (truth == HOWL_ONLY).all(1).mean()
    out = {'clips': f"{len(mix) // per_split:,}",
           'best guess exactly right': f"{right.mean():.1%}",
           'found every labeled sound': f"{(mix >= truth).all(1).mean():.1%}",
           'match with labels (mean)': f"{np.mean(both / either):.1%}",
           'stated confidence (mean)': f"{conf.mean():.1%}",
           'confidence gap (stated - real, points)': f"{100 * (conf.mean() - right.mean()):+.1f}",
           'always guess Howl (no model)': f"{howl:.1%}",
           'model minus always Howl (points)': f"{100 * (right.mean() - howl):+.1f}"}
    for j, t in enumerate(TAGS):
        labeled_n = truth[:, j].sum()
        caught = ((mix[:, j] == 1) & (truth[:, j] == 1)).sum()
        false = ((mix[:, j] == 1) & (truth[:, j] == 0)).sum()
        out[f"{t}: labeled clips caught"] = (f"{caught / labeled_n:.0%} of {labeled_n // per_split}" if labeled_n
                                             else f"- (0 labeled)")
        out[f"{t}: false alarms per 100 clips"] = f"{100 * false / len(mix):.1f}"
    return out


lo, hi = wilson(((mix_test == truth_test).all(1)).sum(), len(test))
split_right = summary20['best guess exactly right']
comparison = pd.DataFrame({
    'training, seen (optimistic)': performance(mix_seen, truth20, conf_seen),
    'training, held out (6 splits)': performance(mix_held, truth_held, conf_held, per_split=len(SPLIT_NAMES)),
    'test (never seen)': performance(mix_test, truth_test, conf_test)})
comparison.loc['exactly right: range'] = ['-', f"{split_right.min():.1%} to {split_right.max():.1%} over the 6 splits",
                                          f"{lo:.1%} to {hi:.1%} (95% range)"]
comparison = comparison.reindex(['clips', 'best guess exactly right', 'exactly right: range'] +
                                [r for r in comparison.index if r not in ('clips', 'best guess exactly right',
                                                                          'exactly right: range')])
print('Training against test: the final tool (per-sound cutoffs and confidence set on training clips only)')
display(comparison)

held_right, test_right = (mix_held == truth_held).all(1).mean(), (mix_test == truth_test).all(1).mean()
verdict = ('inside' if lo <= held_right <= hi else 'outside')
print(f"\nTest: exactly right for {test_right:.1%} of {len(test)} clips (95% range {lo:.1%} to {hi:.1%}); training held out: "
      f"{held_right:.1%}. The held-out result is {verdict} the test's 95% range"
      + (', so the test agrees with what the held-out checks predicted.' if verdict == 'inside'
         else ': the test clips differ from training more than chance explains.'))

#how honest the confidence is on the test clips, and where the model is right
t12 = pd.DataFrame({'right': (mix_test == truth_test).all(1), 'confidence': conf_test,
                    'truth': [mix_name(t) for t in truth_test], 'seen recorder': seen_recorder.values})
t12['band'] = pd.cut(t12.confidence, BANDS20, right=False, labels=BAND_NAMES20)
print('\nTest clips by stated confidence: how often the whole mix is really right')
display(t12.groupby('band', observed=True).agg(clips=('right', 'size'), stated_confidence=('confidence', 'mean'),
                                               really_right=('right', 'mean')).round(3))
print('\nTest clips by true mix (the test labels)')
display(t12.groupby('truth').agg(clips=('right', 'size'), right=('right', 'mean'), confidence=('confidence', 'mean'))
        .sort_values('clips', ascending=False).round(3))
print('\nTest clips from recorders the model heard in training, and from new recorders')
display(t12.groupby(t12['seen recorder'].map({True: 'recorder in training', False: 'new recorder'}))
        .agg(clips=('right', 'size'), right=('right', 'mean'), confidence=('confidence', 'mean')).round(3))

## 10. Chunk tags: teaching the model where each sound is

The spreadsheet labels say which sounds are in a clip, not **when**, so every head learned from whole-clip summaries.
This section lets a reviewer tag individual **3-second chunks** of the training clips, then measures whether training on
those chunk tags makes the chunk-level guesses (and the clip-level best guess) more accurate.

**At a glance**

- **Step 1 (next cell):** `chunk_sheet` writes `chunk_tags_training_YYYY-MM-DD.xlsx`: one row per coyote-moment chunk
  of the 845 training clips, with its time, BirdNET Coyote score, each sound's chance, YipNET's chunk guess (the sounds
  whose chance clears that sound's cutoff) and a blank **Chunk tag** column. A new file name each run; nothing is
  overwritten.
- **Honest chances:** each clip's chunks are scored by models trained **without** that clip's recorder and night
  (the 5 folds of the final split), so the chances on these training clips are as honest as on new recordings.
  They are rough pointers: the heads were trained on whole clips, so a chunk's 70% is not calibrated like a clip's
  confidence.
- **Step 2 (cells after):** after tagging, `chunk_tag_check("the file.xlsx")` compares, on the tagged chunks only,
  the models trained **without** chunk tags (before) and **with** them (after), each chunk judged only by models that
  never saw its clip's recorder and night. It also reports the clip-level best guess before and after, so a gain on
  chunks that costs whole-clip accuracy is visible.
- **Caveats:** tags typed while looking at YipNET's guess can lean toward it; and these training clips cannot also be
  the test of the result (a fresh labeled batch is the clean test).

**Result (Nov 21, 256 tagged chunks from 89 clips; adopted):** held out by recorder and night, the tagged chunks are exactly right 35.5% of the time with the chunk tags against 17.6% without (Bark found in 40 of 108 chunks against 11, Yip-Yap 29 of 76 against 0; Howl false alarms 44 against 111). Whole clips are unchanged within chance: held out 63.2% against 62.6%, test 66.3% against 66.9% (5 test clips better, 6 worse); on the test clips Bark is caught in 11 of 26 clips against 4, with 6 false alarms against 1. Section 9 reports the model without chunk tags.

**Next cell:** defines the chunk tools and writes the chunk sheet (5 trainings: a few minutes).

In [ ]:
import datetime

#Section 10, step 1: the chunk sheet. Every coyote-moment chunk (3 s) of the training clips, with each sound's out-of-fold
#chance, YipNET's chunk guess and a blank Chunk tag column. Tagged chunks become extra training examples (step 2)
MAKE_CHUNK_SHEET = False             # True writes a new, blank chunk sheet on Run All
CHUNK_SHEET_MOMENTS_ONLY = True      # False lists every chunk, including those where BirdNET hears no coyote
CHUNK_DATA_SHEET = 'Chunks'
CHUNK_GUESS_COL = 'YipNET chunk guess'
CHUNK_TAG_COL = ('Chunk tag (the sounds in these 3 s joined by " + ", e.g. Howl + Bark; names: ' + ', '.join(TAGS)
                 + '; "ok" if the chunk guess is right; "none" if no coyote sound)')
FILE_ROW = {os.path.basename(p): i for i, p in enumerate(dat.audio_path)}


def train_with_chunks(clip_rows, chunk_X=None, chunk_truth=None, chunk_owner=None):
    """train_best_guess (the final models' method) on the clips `clip_rows` plus any tagged chunks of those clips. Each
    chunk travels with its clip in every split, so a chunk never helps judge its own clip (or the other way round)."""
    clip_rows = np.asarray(clip_rows)
    if chunk_X is None:
        chunk_X = np.zeros((0, V_birdnet_moments.shape[1]))
        chunk_truth, chunk_owner = np.zeros((0, len(TAGS)), dtype=int), np.zeros(0, dtype=int)
    keep = np.isin(chunk_owner, clip_rows)
    X = np.vstack([V_birdnet_moments[clip_rows], chunk_X[keep]])
    truth = np.vstack([truth20[clip_rows], chunk_truth[keep]])
    owner = np.r_[clip_rows, chunk_owner[keep]]
    rows_of = lambda clip_ids: np.flatnonzero(np.isin(owner, clip_ids))
    folds = [(rows_of(rest), rows_of(held), rows_of(stop)) for rest, held, stop in final_split_folds(clip_rows)]
    fit_clips, stop_clips = [clip_rows[i] for i in group_folds(clip_rows, FINAL_SPLIT, n_splits=5)[0]]
    return train_best_guess(X, truth, rows_of(fit_clips), rows_of(stop_clips), folds)


def fold_models(chunk_X=None, chunk_truth=None, chunk_owner=None):
    """The final split's 5 folds: for each, the held-out clips and a model trained on the other clips (and their
    tagged chunks, if given)."""
    out = []
    for k, (rest, held, _) in enumerate(final_split_folds(ALL_ROWS)):
        out.append((held, train_with_chunks(rest, chunk_X, chunk_truth, chunk_owner)))
        print(f"    fold {k + 1} of 5 trained ({time.time() - started:.0f}s)")
    return out


def oof_chances(models, X, owner):
    """Each row's per-sound chances and cutoffs from the fold model that did not train on its clip."""
    P, cut = np.zeros((len(X), len(TAGS))), np.zeros((len(X), len(TAGS)))
    for held, m in models:
        r = np.isin(owner, held)
        if r.any():
            P[r], cut[r] = m['chance'](X[r]), m['cutoffs']
    return P, cut


def chunk_guess(P, cut):
    """A chunk's sounds (0/1 rows): those whose chance clears the sound's cutoff; Chorus adds Group (and Howl), as in
    the labels."""
    g = (P >= cut).astype(int)
    chorus = g[:, TAGS.index('Chorus')] == 1
    g[chorus, TAGS.index('Group')] = 1
    if CHORUS_INCLUDES_HOWL:
        g[chorus, TAGS.index('Howl')] = 1
    return g


def chunk_name(row):
    return mix_name(row) or 'none'


def chunks_of_clips(rows):
    """The chunks of the clips `rows` (coyote moments only, unless CHUNK_SHEET_MOMENTS_ONLY is False): their
    coyote-moments rows, owner clip, position in the clip and Coyote score."""
    X, owner, pos, coy = [], [], [], []
    for i in rows:
        e, s = clips[i]
        k = np.flatnonzero(coyote_chunks(s['coyote']) if CHUNK_SHEET_MOMENTS_ONLY else np.ones(len(e), bool))
        X.append(chunk_summaries(e, s['coyote'])[k])
        owner.append(np.full(len(k), i))
        pos.append(k)
        coy.append(s['coyote'][k])
    return np.vstack(X), np.concatenate(owner), np.concatenate(pos), np.concatenate(coy)


def write_chunk_sheet(df, file):
    """The chunk sheet: a tidy Chunks sheet (frozen header, filters, the Chunk tag column highlighted, each File linked
    to its WAV) and a How to tag sheet."""
    how = pd.DataFrame({'How to tag': [
        'Each row is one 3-second chunk of a training clip. Click the File name to open the clip, and go to the Start time.',
        'Rows are in clip order. To start with the chunks YipNET is least sure about, sort by "Least sure order" (1 = least sure).',
        'Type the sounds you hear in those 3 seconds in the Chunk tag column, joined by " + " (e.g. Howl + Bark). Names: '
        + ', '.join(TAGS) + '. Group means more than one coyote; Chorus always counts as Group and Howl.',
        'Type "ok" if YipNET\'s chunk guess is right, or "none" if there is no coyote sound in the chunk.',
        'Leave Chunk tag blank for chunks you did not check. Any number of chunks helps; tagging whole clips at a time is best.',
        'Save the file, then in the notebook run chunk_tag_check("this file.xlsx") (Section 10).']})
    with pd.ExcelWriter(file, engine='openpyxl') as xw:
        df.to_excel(xw, sheet_name=CHUNK_DATA_SHEET, index=False)
        how.to_excel(xw, sheet_name='How to tag', index=False)
        ws = xw.sheets[CHUNK_DATA_SHEET]
        for j, header in enumerate(df.columns, start=1):
            cell = ws.cell(row=1, column=j)
            cell.font, cell.fill = _HEAD_FONT, _fill(SHEET_GREEN)
            cell.alignment = Alignment(wrap_text=True, vertical='top')
            width = 60 if header == CHUNK_TAG_COL else min(max(len(_shown(v)) for v in [header[:22]] + list(df[header])) + 2, 40)
            ws.column_dimensions[get_column_letter(j)].width = width
            if header == CHUNK_TAG_COL:
                for r in range(2, len(df) + 2):
                    ws.cell(row=r, column=j).fill = _fill(SHEET_INPUT)
            if header.endswith('chance (%)'):
                for r in range(2, len(df) + 2):
                    ws.cell(row=r, column=j).number_format = '0"%"'
        file_col = list(df.columns).index('File') + 1
        folder = os.path.relpath(os.path.dirname(os.path.abspath(dat.audio_path.iloc[0])), os.path.dirname(os.path.abspath(file)))
        for r in range(2, len(df) + 2):
            cell = ws.cell(row=r, column=file_col)
            cell.hyperlink = os.path.join(folder, str(cell.value)).replace('\\', '/')
            cell.font = Font(name='Calibri', size=11, color='0563C1', underline='single')
        ws.row_dimensions[1].height = 60
        ws.freeze_panes = 'B2'
        ws.auto_filter.ref = ws.dimensions
        xw.sheets['How to tag'].column_dimensions['A'].width = 130


def chunk_sheet(file=None):
    """One row per chunk of the training clips, with out-of-fold chances (see the section notes); written to `file`."""
    global CHUNK_FOLD_MODELS
    X, owner, pos, coy = chunks_of_clips(ALL_ROWS)
    CHUNK_FOLD_MODELS = fold_models()
    P, cut = oof_chances(CHUNK_FOLD_MODELS, X, owner)
    guess = chunk_guess(P, cut)
    #least sure: the sound whose chance is closest to its cutoff (on the log-odds scale), smallest gap first
    gap = np.abs(logit(P) - logit(cut)).min(1)
    files = np.array([os.path.basename(p) for p in dat.audio_path])
    df = pd.DataFrame({'File': files[owner],
                       'Clip labels (spreadsheet)': [chunk_name(truth20[i]) for i in owner],
                       'Chunk': pos + 1,
                       'Time': [seconds_label(p) for p in pos],
                       'Start (s)': (pos * CHUNK_SECONDS).round(0).astype(int),
                       'Coyote score': coy.astype(float).round(3)})
    for j, t in enumerate(TAGS):
        df[f'{t} chance (%)'] = (100 * P[:, j]).round(1)
    df[CHUNK_GUESS_COL] = [chunk_name(g) for g in guess]
    df['Least sure order'] = pd.Series(gap).rank(method='first').astype(int).values
    df[CHUNK_TAG_COL] = ''
    if file:
        write_chunk_sheet(df, file)
    return df


started = time.time()
if MAKE_CHUNK_SHEET:
    stem = f"chunk_tags_training_{datetime.date.today():%Y-%m-%d}"
    CHUNK_SHEET_FILE, n = f"{stem}.xlsx", 1
    while os.path.exists(CHUNK_SHEET_FILE):
        n += 1
        CHUNK_SHEET_FILE = f"{stem}_{n}.xlsx"
    chunks20 = chunk_sheet(CHUNK_SHEET_FILE)
    print(f"{len(chunks20)} chunks from {chunks20.File.nunique()} training clips ({time.time() - started:.0f}s)")
    print('Chunk sheet saved to:', os.path.abspath(CHUNK_SHEET_FILE))
    print('\nYipNET chunk guesses (most common):')
    print(chunks20[CHUNK_GUESS_COL].value_counts().head(8).to_string())
    display(chunks20.head(10))

**Next cell:** defines `chunk_tag_check`, which reads the tagged chunks and compares training without and with them.

In [ ]:
#Section 10, step 2: read the Chunk tag column of one or more chunk sheets (a later file wins for the same chunk), then
#compare the fold models trained WITHOUT chunk tags (before) and WITH the tagged chunks of their training clips (after).
#Each tagged chunk is judged only by models that never saw its clip's recorder and night, so the comparison is held out.
#Returns the final model trained on all 845 clips plus every tagged chunk; it is used only if ADOPT_CHUNK_MODEL is True
def read_chunk_tags(*files):
    """The tagged chunks: coyote-moments rows, 0/1 truth over TAGS, owner clip and position."""
    tags = {}
    for file in files:
        try:
            sheet = pd.read_excel(file, sheet_name=CHUNK_DATA_SHEET)
        except PermissionError:
            raise PermissionError(f"{file} is open in Excel: save and close it, then run this cell again")
        tag_col = next(c for c in sheet.columns if str(c).startswith('Chunk tag'))
        for _, r in sheet.iterrows():
            text = str(r[tag_col]).strip()
            if text in ('', 'nan'):
                continue
            if text.lower() == 'ok':
                text = str(r[CHUNK_GUESS_COL])
            truth = np.zeros(len(TAGS), dtype=int) if text.lower() == 'none' else parse_mix(text, None)
            tags[(r['File'], int(r['Chunk']) - 1)] = truth
    X, truth, owner, pos = [], [], [], []
    for (f, k), t in tags.items():
        i = FILE_ROW[f]
        e, s = clips[i]
        X.append(chunk_summaries(e, s['coyote'])[k])
        truth.append(t)
        owner.append(i)
        pos.append(k)
    return np.array(X), np.array(truth, dtype=int), np.array(owner), np.array(pos)


def chunk_scores(models, X, truth, owner):
    """Held-out chunk guesses for the tagged chunks and clip best guesses for all 845 clips."""
    P, cut = oof_chances(models, X, owner)
    g = chunk_guess(P, cut)
    clip_right = np.zeros(len(dat), dtype=bool)
    for held, m in models:
        clip_right[held] = (best_mix(m['chance'](V_birdnet_moments[held]), m['cutoffs'])[0] == truth20[held]).all(1)
    return g, (g == truth).all(1), clip_right


def chunk_tag_check(*files):
    global started
    started = time.time()
    X, truth, owner, pos = read_chunk_tags(*files)
    if not len(X):
        print('No Chunk tag filled in yet: nothing to learn.')
        return None
    print(f"{len(X)} tagged chunks from {len(np.unique(owner))} training clips")
    print('Training without chunk tags (before):')
    before_models = globals().get('CHUNK_FOLD_MODELS') or fold_models()
    print('Training with the tagged chunks (after):')
    after_models = fold_models(X, truth, owner)
    g0, right0, clip0 = chunk_scores(before_models, X, truth, owner)
    g1, right1, clip1 = chunk_scores(after_models, X, truth, owner)
    n = len(X)
    print(f"\nTagged chunks exactly right (held out): before {right0.sum()} of {n} ({right0.mean():.1%}), "
          f"after {right1.sum()} of {n} ({right1.mean():.1%})")
    print(f"  wrong before and right after: {(~right0 & right1).sum()}; right before and wrong after: {(right0 & ~right1).sum()}")
    rows = []
    for j, t in enumerate(TAGS):
        has = truth[:, j] == 1
        rows.append({'sound': t, 'tagged chunks with it': int(has.sum()),
                     'caught before': int((g0[has, j] == 1).sum()), 'caught after': int((g1[has, j] == 1).sum()),
                     'false alarms before': int((g0[~has, j] == 1).sum()), 'false alarms after': int((g1[~has, j] == 1).sum())})
    print('\nEach sound on the tagged chunks (held out)')
    display(pd.DataFrame(rows).set_index('sound'))
    print(f"Whole clips, best guess exactly right (held out, all {len(dat)} clips): before {clip0.mean():.1%}, "
          f"after {clip1.mean():.1%}")
    print('With a few hundred tagged chunks, differences of a few points are within chance; the clean test of the '
          'result is a fresh labeled batch.')
    print('\nTraining the final model on all clips plus every tagged chunk:')
    final_chunks = train_with_chunks(ALL_ROWS, X, truth, owner)
    print(f"done ({time.time() - started:.0f}s)")
    return {'final model': final_chunks, 'chunks right before': right0, 'chunks right after': right1,
            'clips right before': clip0, 'clips right after': clip1}

**Next cell:** runs `chunk_tag_check` on the tagged chunk sheets in `CHUNK_TAG_FILES`, shows the model trained with the chunk tags on the test clips of Section 9 (score only), and, with `ADOPT_CHUNK_MODEL = True`, makes it the final tool (`final20`). The demo-app save and the best-guess sheet come after this section, so they use the adopted model.

In [ ]:
#the tagged chunk sheets (a later file wins for the same chunk). Tags typed after this run count on the next Run All
CHUNK_TAG_FILES = ['chunk_tags_pilot_2026-11-20_2.xlsx']
#True replaces the final tool with the model trained with the chunk tags. Adopted Nov 21 (team decision): whole clips are
#unchanged within chance, and the sounds inside each clip (Bark, Yip-Yap) are found far more often
ADOPT_CHUNK_MODEL = True

final20_without_chunks = final20
#a sheet that is not in this folder is skipped (the chunk-tag sheet is the team's own data, not on GitHub); without
#any, the final tool stays the model without chunk tags
missing_chunk_sheets = [f for f in CHUNK_TAG_FILES if not os.path.exists(f)]
if missing_chunk_sheets:
    print('Not found, skipped:', missing_chunk_sheets)
    CHUNK_TAG_FILES = [f for f in CHUNK_TAG_FILES if os.path.exists(f)]
if CHUNK_TAG_FILES:
    chunk_result = chunk_tag_check(*CHUNK_TAG_FILES)
    if chunk_result is not None:
        #the chunk model on Section 9's test clips, for comparison only (the test labels still only score)
        X_test12 = stored['summary'][[row_of[f] for f in test.File]].astype(float)
        rows = {}
        for name, m in (('without chunk tags (Section 9)', final20_without_chunks),
                        ('with chunk tags', chunk_result['final model'])):
            mix_t, raw_t = best_mix(m['chance'](X_test12), m['cutoffs'])
            right_t = (mix_t == truth_test).all(1)
            rows[name] = {'best guess exactly right': f"{right_t.mean():.1%}",
                          'found every labeled sound': f"{(mix_t >= truth_test).all(1).mean():.1%}",
                          'stated confidence (mean)': f"{m['recal'](raw_t).mean():.1%}",
                          **{f"{t}: caught / false alarms": f"{int(((mix_t[:, j] == 1) & (truth_test[:, j] == 1)).sum())} of "
                             f"{int(truth_test[:, j].sum())} / {int(((mix_t[:, j] == 1) & (truth_test[:, j] == 0)).sum())}"
                             for j, t in enumerate(TAGS) if t in ('Bark', 'Yip-Yap', 'Group')}}
        print(f"\nTest clips ({len(test)}, never trained on): final tool without and with the chunk tags")
        display(pd.DataFrame(rows))
        if ADOPT_CHUNK_MODEL:
            final20 = chunk_result['final model']
            print('final20 is now the model trained with the chunk tags (used by the demo-app save and the sheet below)')
else:
    print('No tagged chunk sheets listed in CHUNK_TAG_FILES yet.')


## 11. Test #2: a second set of new labeled clips

**At a glance**

- **Question:** Does the test result of Section 9 hold on a second, independent batch, and does the model trained with
  the chunk tags (Section 10) do better than the model without them?
- **Method:** The 181 clips of `labels_test2.xlsx` (audio in `test2/Test_2_Audio_Files/`).
  Labels are cleaned and turned into mixes exactly as in Section 9. Clips also in the training data are left out. Both
  final tools tag the clips: without chunk tags (Section 9's model) and with them (the final tool since Section 10).
  The test labels are used only to score; nothing is tuned on them.
- **Result (run of Nov 21):** 180 clips (1 was also in training). Exactly right: **63.3%** without chunk tags (95% range
  56.1% to 70.0%) and **61.7%** with them, against 56.7% for always guessing Howl; the difference is within chance (9
  clips better without, 6 with). With Test #1, 361 clips: 65.1% and 64.0%. Both agree with the held-out 61.9% of
  Section 8 (6 splits). With the chunk tags, Yip-Yap is caught in 6 of 14 clips (2 without; false alarms 8 against 1) but Bark is
  not (4 of 23 against 3: Test #1's gain did not repeat), and pure howls are right 88% of the time (96% without). The
  confidence is more honest with the chunk tags (stated 63.0%, right 61.7%; without: 67.8% and 63.3%), and the clips
  under 50% confidence hold 32 of the 69 wrong guesses (without: 25 of 66). Every Test #2 clip comes from a recorder
  that is also in the training data, so this tests new recordings, not new places.
- **Output:** a test best-guess sheet made with the final tool, `TEST2_best_guesses_YYYY-MM-DD.xlsx` (a new name each
  run, never overwritten), and a table comparing the two models.

**Next cell:** reads the Test #2 labels and makes its best-guess sheet (BirdNET and the howl pitch tracker run on the
test clips: about 10 minutes).

In [ ]:
import datetime

#Test #2: the labeled clips of the second test spreadsheet, scored like Section 9. Clips also in the training data are
#left out. The sheet is made with the final tool (with the chunk tags); the next cell also scores the model without them
TEST2_FOLDER = 'test2'
TEST2_LABELS_FILE = os.path.join(TEST2_FOLDER, 'labels_test2.xlsx')
TEST2_AUDIO_DIR = os.path.join(TEST2_FOLDER, 'Test_2_Audio_Files')

#Section 11 runs only when the Test #2 labels and audio are in this folder (team data, not on GitHub)
TEST2_AVAILABLE = os.path.exists(TEST2_LABELS_FILE) and os.path.isdir(TEST2_AUDIO_DIR)
if not TEST2_AVAILABLE:
    print('Test #2 data not found: Section 11 skipped.')
else:
    test2 = pd.read_excel(TEST2_LABELS_FILE)
    tidy = lambda s: s.astype(object).where(s.isna(), s.astype(str).str.strip())
    for column in LABEL_COLUMNS:
        test2[column] = tidy(test2[column]).map(lambda v: v.title() if isinstance(v, str) else v)
    test2['ARU or Location'] = tidy(test2['ARU or Location'])
    test2['File'] = test2['Original File'].astype(str).str.strip()
    has_audio2 = test2.File.map(lambda f: os.path.exists(os.path.join(TEST2_AUDIO_DIR, f)))
    print(f"{has_audio2.sum()} of {len(test2)} Test #2 clips found in {TEST2_AUDIO_DIR}/")
    test2 = test2[has_audio2].sort_values('File').reset_index(drop=True)
    in_training2 = test2.File.isin({os.path.basename(p) for p in dat.audio_path})
    print(f"{in_training2.sum()} Test #2 clip(s) also in the training data, left out: {list(test2.File[in_training2])}")
    test2 = test2[~in_training2].reset_index(drop=True)
    unknown2 = test2[LABEL_COLUMNS].stack()
    unknown2 = unknown2[~unknown2.isin(CALLS)]
    if len(unknown2):
        print('Label names that are not one of the seven sounds (left out, as in training):', unknown2.value_counts().to_dict())

    #each clip's true mix, built exactly as for the training clips (Section 6)
    any_group2 = test2['Group'].notna() | (test2[LABEL_COLUMNS] == 'Chorus').any(axis=1)
    truth_test2 = np.array([[(tag == 'Group' and group) or (tag != 'Group' and tag in set(calls)) for tag in TAGS]
                            for calls, group in zip(test2[LABEL_COLUMNS].values, any_group2)], dtype=int)
    if CHORUS_INCLUDES_HOWL:
        truth_test2[truth_test2[:, TAGS.index('Chorus')] == 1, TAGS.index('Howl')] = 1
    TEST2_LABELS = {f: mix_name(t) for f, t in zip(test2.File, truth_test2)}
    seen2 = test2['ARU or Location'].isin(set(dat['ARU or Location']))
    print(f"Test #2 clips from recorders that are also in the training data: {seen2.sum()} of {len(test2)}")
    print('\nTest #2 clips with each sound present')
    print(pd.Series(truth_test2.sum(0), index=TAGS).to_string())

    stem = f"TEST2_best_guesses_{datetime.date.today():%Y-%m-%d}"
    TEST2_SHEET_FILE, n = f"{stem}.xlsx", 1
    while os.path.exists(TEST2_SHEET_FILE) or os.path.exists(os.path.splitext(TEST2_SHEET_FILE)[0] + '_birdnet.npz'):
        n += 1
        TEST2_SHEET_FILE = f"{stem}_{n}.xlsx"
    print()
    test2_sheet = best_guess_sheet(TEST2_AUDIO_DIR, file=TEST2_SHEET_FILE, files=list(test2.File), labels=TEST2_LABELS,
                                   title='TEST #2 DATA: batch summary', labels_source=TEST2_LABELS_FILE)
    print('Test #2 spreadsheet saved to:', os.path.abspath(TEST2_SHEET_FILE))

**Next cell:** both final tools on Test #2 (and on Test #1 again, for the two tests together): exactly right with its 95% range, each sound, how honest the confidence is, and the clips each model gets right that the other misses.

In [ ]:
#both final tools on the Test #2 clips, from the BirdNET numbers saved with the Test #2 sheet (unrounded); the sheet
#itself was made with the final tool, so its best guesses must match
from scipy.stats import binomtest

if not TEST2_AVAILABLE:
    print('Test #2 data not found: Section 11 skipped.')
else:

    stored2 = np.load(os.path.splitext(TEST2_SHEET_FILE)[0] + '_birdnet.npz', allow_pickle=True)
    row_of2 = {f: i for i, f in enumerate(stored2['files'])}
    X_test2 = stored2['summary'][[row_of2[f] for f in test2.File]].astype(float)
    X_test1 = stored['summary'][[row_of[f] for f in test.File]].astype(float)


    def wilson(k, n, z=1.96):
        """95% range for a share of k out of n (Wilson score interval)."""
        p, z2 = k / n, z * z
        centre, half = (p + z2 / (2 * n)) / (1 + z2 / n), z * np.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / (1 + z2 / n)
        return centre - half, centre + half


    TWO_MODELS = {'without chunk tags': final20_without_chunks, 'with chunk tags (final tool)': final20}
    HOWL_ONLY = np.array([t == 'Howl' for t in TAGS], dtype=int)
    rows, right2 = {}, {}
    for tname, X, truth in (('Test #2', X_test2, truth_test2), ('Test #1', X_test1, truth_test)):
        for name, m in TWO_MODELS.items():
            mix, raw = best_mix(m['chance'](X), m['cutoffs'])
            conf = m['recal'](raw)
            right = (mix == truth).all(1)
            right2[tname, name] = right
            if tname == 'Test #2' and m is final20:
                assert (test2_sheet.set_index('File').loc[test2.File, 'Best guess'].values == [mix_name(g) for g in mix]).all()
            lo, hi = wilson(right.sum(), len(right))
            pure = (truth == HOWL_ONLY).all(1)
            row = {'clips': len(right), 'best guess exactly right': f"{right.mean():.1%}",
                   'exactly right: 95% range': f"{lo:.1%} to {hi:.1%}",
                   'always guess Howl (no model)': f"{pure.mean():.1%}",
                   'found every labeled sound': f"{(mix >= truth).all(1).mean():.1%}",
                   'stated confidence (mean)': f"{conf.mean():.1%}",
                   'pure Howl clips right': f"{right[pure].mean():.1%}",
                   'clips under 50% confidence': int((conf < 0.5).sum()),
                   'wrong guesses among them': f"{int((~right & (conf < 0.5)).sum())} of {int((~right).sum())}"}
            for j, t in enumerate(TAGS):
                if t in ('Bark', 'Yip-Yap', 'Chorus', 'Group'):
                    row[f"{t}: caught"] = f"{int(((mix[:, j] == 1) & (truth[:, j] == 1)).sum())} of {int(truth[:, j].sum())}"
                    row[f"{t}: false alarms"] = int(((mix[:, j] == 1) & (truth[:, j] == 0)).sum())
            rows[f"{tname}, {name}"] = row
    display(pd.DataFrame(rows))

    a, b = [np.r_[right2['Test #2', n], right2['Test #1', n]] for n in TWO_MODELS]
    for label, x, y in (('Test #2', right2['Test #2', 'without chunk tags'], right2['Test #2', 'with chunk tags (final tool)']),
                        (f'Both tests ({len(a)} clips)', a, b)):
        k, n = int((~x & y).sum()), int((x ^ y).sum())
        print(f"{label}: exactly right {x.mean():.1%} without chunk tags, {y.mean():.1%} with; right only without: "
              f"{int((x & ~y).sum())}, only with: {k} (sign test p = {binomtest(k, n).pvalue:.2f}: "
              f"{'within chance' if binomtest(k, n).pvalue > 0.05 else 'a real difference'})")

### Save the final tool for the demo app

**Next cell:** saves the final models (with the chunk tags of Section 10) and the functions above to `demo_app/final_tool_local.pkl` for the demo app
(`demo_app/app.py`, a web page on this computer where clips or folders are dragged in and analyzed with this exact
model). The app checks on start-up that it gives the notebook's answers for the two made-up sounds. Rerun this cell
whenever the final models change. The file holds the training clips' BirdNET numbers, so it stays off GitHub (`.gitignore`); the app uses it instead of the public `demo_app/final_tool.pkl` (trained models only) when it exists.

In [ ]:
#the demo app (demo_app/app.py: a small web page on this computer) runs the exact final tool of this
#notebook: these final models (final20, final_checks) and these functions, unchanged. This cell saves them to
#demo_app/final_tool.pkl with cloudpickle, which stores functions defined in a notebook by value. It also stores the
#best-guess sheet of the two made-up sounds, so the app checks on start-up that it gives the notebook's answers.
#The file holds the training clips' BirdNET numbers (for the review checks), so, like birdnet_features.npz, it never
#goes on GitHub. Rerun this cell whenever the final models change (for example after learn_from_corrections).
#Needs cloudpickle and flask (requirements.txt)
import platform
import sklearn
import cloudpickle

DEMO_DIR = 'demo_app'
#saved as final_tool_local.pkl: it holds your training recordings' BirdNET numbers, so it is never committed
#(.gitignore); the app uses it instead of the public demo_app/final_tool.pkl when it exists
DEMO_BUNDLE = os.path.join(DEMO_DIR, 'final_tool_local.pkl')
os.makedirs(DEMO_DIR, exist_ok=True)

#the notebook's own answers for the made-up sounds, for the app's start-up check
demo_reference = best_guess_sheet('demo_sounds')

FINAL_TOOL = {
    #where the models come from
    'saved': f"{datetime.datetime.now():%Y-%m-%d %H:%M}", 'training clips': len(dat),
    'python': platform.python_version(), 'sklearn': sklearn.__version__,
    #BirdNET exactly as in Section 2
    'birdnet load': {'args': ('acoustic', '2.4', 'tf'), 'kwargs': {'library': 'litert'}},
    'KEPT_SPECIES': KEPT_SPECIES,
    #the settings and sheet layout of Section 8
    'TAGS': TAGS, 'CHUNK_SECONDS': CHUNK_SECONDS, 'NO_COYOTE_BELOW': NO_COYOTE_BELOW,
    'TIME_CHANCE': TIME_CHANCE, 'LOOSE_TIME_CHANCE': LOOSE_TIME_CHANCE, 'SHEET_BANDS': SHEET_BANDS,
    'STRICT_TIMES_COL': STRICT_TIMES_COL, 'LOOSE_TIMES_COL': LOOSE_TIMES_COL, 'CORRECTED_COL': CORRECTED_COL,
    'HOW_TO_CORRECT': HOW_TO_CORRECT,
    #the final models trained on all the training clips, and the review checks for new recordings
    'model': final20, 'checks': final_checks,
    #the functions best_guess_sheet uses
    'functions': {'coyote_moments': coyote_moments, 'coyote_chunks': coyote_chunks, 'best_mix': best_mix,
                  'mix_name': mix_name, 'sound_times': sound_times, 'chunk_summaries': chunk_summaries,
                  'howl_columns': howl_columns, 'howl_spans': howl_spans, 'seconds_label': seconds_label,
                  'write_best_guess_workbook': write_best_guess_workbook},
    'reference': demo_reference, 'reference folder': 'demo_sounds',
}
with open(DEMO_BUNDLE, 'wb') as f:
    cloudpickle.dump(FINAL_TOOL, f)
print(f"Saved {DEMO_BUNDLE} ({os.path.getsize(DEMO_BUNDLE) / 1e6:.0f} MB): final models trained on {len(dat)} clips, "
      f"cutoffs (%) { {t: int(round(100 * c)) for t, c in zip(TAGS, final20['cutoffs'])} }")
print('Start the demo from this folder with:  python demo_app/app.py   (then open http://127.0.0.1:5000)')

## Make the best-guess spreadsheet

**Next cell:** runs the final tool (with the chunk tags of Section 10) on every recording in `RECORDINGS_FOLDER` and saves the spreadsheet next to this
notebook. Each run makes a new file named with today's date (`best_guesses_YYYY-MM-DD.xlsx`, then `_2`, `_3`, ...), so it
never overwrites a sheet the reviewers may already have corrected.

In [ ]:
import datetime

#the folder of recordings to tag: 'audio' is the 845 training clips; point it at a folder of new recordings
#for an honest test (the confidences on the training clips are optimistic)
RECORDINGS_FOLDER = 'audio'

#an earlier best-guess sheet of the same recordings: its longest-howl measurements are reused, which skips the slow
#pitch tracking (None measures every howl again)
REUSE_SHEET = 'best_guesses_2026-11-14_5.xlsx'
if RECORDINGS_FOLDER != 'audio' or not os.path.exists(REUSE_SHEET):
    REUSE_SHEET = None

#a new file name each run, so an existing (possibly corrected) sheet is never overwritten
stem = f"best_guesses_{datetime.date.today():%Y-%m-%d}"
SHEET_FILE, n = f"{stem}.xlsx", 1
while os.path.exists(SHEET_FILE) or os.path.exists(os.path.splitext(SHEET_FILE)[0] + '_birdnet.npz'):
    n += 1
    SHEET_FILE = f"{stem}_{n}.xlsx"

sheet = best_guess_sheet(RECORDINGS_FOLDER, file=SHEET_FILE, reuse=REUSE_SHEET)
print('Spreadsheet saved to:', os.path.abspath(SHEET_FILE))
display(sheet.head(10))